# Extended calmodulin surface analysis

Apo 1CFD and holo-derived 1CLL (protein heavy atoms; calcium omitted). Legacy aperiodic `Maps`/`ionic` workflow, 52 radial AC coefficients plus two physicochemical proxies. All inputs are bundled; no downloads. Results are written to a fresh directory.

Classification uses training-fold scaling. Point-level splits, in-sample permutation importance, LDA separation and KS tests remain exploratory diagnostics for these two structures; they do not validate prediction on independent proteins. Separate UMAP embeddings have separate coordinate bases. Graph connectivity describes geometric neighbors, not energetic transitions.


In [ ]:
from pathlib import Path
import os, sys, datetime
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/mapsy').is_dir() and (p / 'examples').is_dir())
OUTPUT = Path(os.environ.get('MAPSY_EXTENDED_OUTPUT', ROOT / 'examples/results' / ('extended_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))))
OUTPUT.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
_original_show = plt.show
_figure_number = 0
def save_visible_figures(*args, **kwargs):
    global _figure_number
    for num in plt.get_fignums():
        _figure_number += 1
        plt.figure(num).savefig(OUTPUT / f'figure_{_figure_number:03d}.png', dpi=120, bbox_inches='tight')
    plt.close('all')
plt.show = save_visible_figures
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
import mapsy
print('PACKAGE:', mapsy.__file__, '\nOUTPUT:', OUTPUT)


# MapSy Extended Descriptors: Electrostatics + Hydrophobicity

Adds two physicochemical descriptor channels onto existing MapSy CS points:

1. **Electrostatic potential** — distance-weighted sum of AMBER ff14SB partial charges
2. **Hydrophobicity field** — distance-weighted Kyte-Doolittle scale

Both are interpolated onto contact space points using inverse-distance weighting
within a cutoff radius. They plug directly into the existing Maps/descriptor pipeline
as additional columns in `maps.data`.

Run after generating `cs` and `maps` objects from the standard pipeline.


---
## Section 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import KDTree
from scipy.stats import ks_2samp, mannwhitneyu
from Bio.PDB import PDBParser, DSSP
import urllib.request
import os
import warnings
# Keep warnings visible during verification.

from ase.io import read
from mapsy.data import Grid, System
from mapsy.io.parser import ContactSpaceGenerator, ContactSpaceModel
from mapsy.symfunc.input import SymmetryFunctionsModel, SymFuncModel
from mapsy.symfunc.parser import SymmetryFunctionsParser
from mapsy.maps import Maps

import mapsy.boundaries.ionic as ionic_mod

CPK = {'C':'#909090','N':'#4169E1','O':'#FF4500','S':'#DAA520','CA':'#80B0FF'}
def _atom_color(s): return CPK.get(s, '#AAAAAA')

print('Imports OK ✓')


---
## Section 2 — Physicochemical Lookup Tables

In [ ]:
# Kyte-Doolittle hydrophobicity scale (per residue)
# Range: -4.5 (most hydrophilic) to +4.5 (most hydrophobic)
KD_SCALE = {
    'ALA': 1.8, 'ARG': -4.5, 'ASN': -3.5, 'ASP': -3.5, 'CYS':  2.5,
    'GLN':-3.5, 'GLU': -3.5, 'GLY': -0.4, 'HIS': -3.2, 'ILE':  4.5,
    'LEU': 3.8, 'LYS': -3.9, 'MET':  1.9, 'PHE':  2.8, 'PRO': -1.6,
    'SER':-0.8, 'THR': -0.7, 'TRP': -0.9, 'TYR': -1.3, 'VAL':  4.2,
}

# AMBER ff14SB partial charges — (resname, atomname)
# '*' matches any residue (backbone atoms)
PARTIAL_CHARGES = {
    # Backbone — shared across all residues
    ('*','N' ): -0.4157, ('*','H' ):  0.2719,
    ('*','CA'):  0.0337, ('*','HA'):  0.0823,
    ('*','C' ):  0.5973, ('*','O' ): -0.5679,
    # ALA
    ('ALA','CB'): -0.1825,
    # ARG
    ('ARG','CB'): -0.0007, ('ARG','CG'):  0.0390, ('ARG','CD'):  0.0486,
    ('ARG','NE'): -0.5295, ('ARG','CZ'):  0.8076,
    ('ARG','NH1'): -0.8627, ('ARG','NH2'): -0.8627,
    # ASP
    ('ASP','CB'): -0.0303, ('ASP','CG'):  0.7994,
    ('ASP','OD1'): -0.8014, ('ASP','OD2'): -0.8014,
    # GLU
    ('GLU','CB'): -0.0642, ('GLU','CG'):  0.0136,
    ('GLU','CD'):  0.8054, ('GLU','OE1'): -0.8188, ('GLU','OE2'): -0.8188,
    # LYS
    ('LYS','CB'): -0.0094, ('LYS','CG'):  0.0187, ('LYS','CD'):  0.0271,
    ('LYS','CE'):  0.0994, ('LYS','NZ'): -0.3854,
    # HIS
    ('HIS','CB'): -0.0414, ('HIS','CG'):  0.1880, ('HIS','ND1'): -0.5432,
    ('HIS','CD2'):  0.1046, ('HIS','CE1'):  0.2057, ('HIS','NE2'): -0.2786,
    # PHE
    ('PHE','CB'): -0.0343, ('PHE','CG'):  0.0118,
    ('PHE','CD1'): -0.1256, ('PHE','CD2'): -0.1256,
    ('PHE','CE1'): -0.1704, ('PHE','CE2'): -0.1704, ('PHE','CZ'): -0.1072,
    # TYR
    ('TYR','CB'): -0.0152, ('TYR','CG'): -0.0011,
    ('TYR','CD1'): -0.1906, ('TYR','CD2'): -0.1906,
    ('TYR','CE1'): -0.2341, ('TYR','CE2'): -0.2341,
    ('TYR','CZ'):  0.3226,  ('TYR','OH'): -0.5579,
    # SER
    ('SER','CB'):  0.2117, ('SER','OG'): -0.6546,
    # THR
    ('THR','CB'):  0.3654, ('THR','OG1'): -0.6761, ('THR','CG2'): -0.2438,
    # CYS
    ('CYS','CB'): -0.1231, ('CYS','SG'): -0.3119,
    # MET
    ('MET','CB'): -0.0269, ('MET','CG'):  0.1024,
    ('MET','SD'): -0.2737, ('MET','CE'): -0.0536,
    # ASN
    ('ASN','CB'): -0.2041, ('ASN','CG'):  0.7130,
    ('ASN','OD1'): -0.5931, ('ASN','ND2'): -0.9191,
    # GLN
    ('GLN','CB'): -0.0536, ('GLN','CG'):  0.0651,
    ('GLN','CD'):  0.6951, ('GLN','OE1'): -0.6086, ('GLN','NE2'): -0.9407,
    # TRP
    ('TRP','CB'): -0.0050, ('TRP','CG'): -0.1415, ('TRP','CD1'): -0.1638,
    ('TRP','CD2'):  0.1243, ('TRP','NE1'): -0.3418, ('TRP','CE2'):  0.1380,
    ('TRP','CE3'): -0.2387, ('TRP','CZ2'):  0.0046,
    ('TRP','CZ3'): -0.1961, ('TRP','CH2'): -0.1134,
    # ILE
    ('ILE','CB'):  0.1303, ('ILE','CG1'): -0.0430,
    ('ILE','CG2'): -0.3204, ('ILE','CD1'): -0.0660,
    # LEU
    ('LEU','CB'): -0.1102, ('LEU','CG'):  0.3531,
    ('LEU','CD1'): -0.4121, ('LEU','CD2'): -0.4121,
    # VAL
    ('VAL','CB'):  0.2985, ('VAL','CG1'): -0.3192, ('VAL','CG2'): -0.3192,
    # PRO
    ('PRO','CB'): -0.0070, ('PRO','CG'):  0.0189, ('PRO','CD'):  0.0192,
    # GLY — no side chain
}

def get_charge(resname, atomname):
    """Look up AMBER partial charge. Falls back to backbone '*' then 0."""
    if (resname, atomname) in PARTIAL_CHARGES:
        return PARTIAL_CHARGES[(resname, atomname)]
    if ('*', atomname) in PARTIAL_CHARGES:
        return PARTIAL_CHARGES[('*', atomname)]
    return 0.0  # unknown atom — neutral fallback

print(f'KD scale: {len(KD_SCALE)} residues')
print(f'Partial charges: {len(PARTIAL_CHARGES)} entries')
print(f'Test ASP OD1: {get_charge("ASP", "OD1"):.4f} (expect -0.8014)')
print(f'Test backbone N: {get_charge("ALA", "N"):.4f} (expect -0.4157)')


---
## Section 3 — Core Interpolation Engine

In [ ]:
from Bio.PDB import PDBParser
import numpy as np
from scipy.spatial import KDTree

def interpolate_to_cs(atom_positions, atom_values, cs_positions,
                      r_cut=5.0, default=0.0, method='idw'):
    tree         = KDTree(atom_positions)
    result       = np.full(len(cs_positions), default, dtype=np.float64)
    indices_list = tree.query_ball_point(cs_positions, r_cut)
    for i, neighbors in enumerate(indices_list):
        if not neighbors:
            continue
        neighbors = np.array(neighbors)
        dists     = np.linalg.norm(atom_positions[neighbors] - cs_positions[i], axis=1)
        dists     = np.maximum(dists, 0.1)
        if method == 'idw':
            weights  = 1.0 / dists
            result[i] = np.dot(weights, atom_values[neighbors]) / weights.sum()
        elif method == 'sum':
            result[i] = np.sum(atom_values[neighbors] / dists)
    return result

def build_atom_arrays_from_ase(atoms_ase, pdb_path, model_index=0):
    from pathlib import Path
    records = []
    for line in Path(pdb_path).read_text().splitlines():
        if line.startswith('ENDMDL'): break
        if line.startswith('ATOM  '):
            records.append((line[17:20].strip(), line[12:16].strip(), [float(line[30:38]),float(line[38:46]),float(line[46:54])]))
    raw = np.asarray([r[2] for r in records])
    positions = atoms_ase.get_positions()
    assert raw.shape == positions.shape
    assert np.allclose(positions, raw - raw.min(axis=0) + 8.0, rtol=0, atol=1e-10), 'PDB/ASE atom order differs'
    charges = np.array([get_charge(res,name) for res,name,xyz in records])
    hydrophobics = np.array([KD_SCALE.get(res,0.) for res,name,xyz in records])
    print('Exact PDB/ASE property alignment:',len(records),'records')
    return positions, charges, hydrophobics

def add_physicochemical_descriptors(maps_obj, atoms_ase, pdb_path,
                                     model_index=0,
                                     r_cut_elec=8.0, r_cut_hydro=5.0):
    positions, charges, hydrophobics = build_atom_arrays_from_ase(
        atoms_ase, pdb_path, model_index
    )
    cs_xyz = maps_obj.data[['x','y','z']].values
    print(f'  Atom positions range: x=[{positions[:,0].min():.1f}, {positions[:,0].max():.1f}]'
          f' y=[{positions[:,1].min():.1f}, {positions[:,1].max():.1f}]'
          f' z=[{positions[:,2].min():.1f}, {positions[:,2].max():.1f}]')
    print(f'  CS points range:      x=[{cs_xyz[:,0].min():.1f}, {cs_xyz[:,0].max():.1f}]'
          f' y=[{cs_xyz[:,1].min():.1f}, {cs_xyz[:,1].max():.1f}]'
          f' z=[{cs_xyz[:,2].min():.1f}, {cs_xyz[:,2].max():.1f}]')
    print(f'  Computing electrostatic ({len(positions)} atoms → {len(cs_xyz)} CS points)...')
    elec  = interpolate_to_cs(positions, charges, cs_xyz,
                               r_cut=r_cut_elec, default=0.0, method='sum')
    print(f'  Computing hydrophobicity...')
    hydro = interpolate_to_cs(positions, hydrophobics, cs_xyz,
                               r_cut=r_cut_hydro, default=0.0, method='idw')
    maps_obj.data['electrostatic']  = elec
    maps_obj.data['hydrophobicity'] = hydro
    print(f'  Electrostatic:  [{elec.min():.4f}, {elec.max():.4f}]  std={elec.std():.4f}')
    print(f'  Hydrophobicity: [{hydro.min():.4f}, {hydro.max():.4f}]  std={hydro.std():.4f}')
    if elec.std() < 1e-6:
        print('  WARNING: std≈0 — coordinate frame mismatch')
    if hydro.std() < 1e-6:
        print('  WARNING: std≈0 — coordinate frame mismatch')
    return maps_obj

print('All physicochemical functions defined ✓')

---
## Section 4 — Load Structures and Run Pipeline

In [ ]:
from Bio.PDB import PDBParser, PDBIO, Select
import tempfile, os
import urllib.request
import numpy as np
from ase.io import read

APO_PDB  = '1CFD' # calcium free (apo) calmodulin
HOLO_PDB = '1CLL' # calcium bound (holo) calmodulin

PDB_PATHS = {p: ROOT / 'examples/data/proteins' / f'{p}_protein.pdb' for p in [APO_PDB, HOLO_PDB]}
assert all(p.is_file() for p in PDB_PATHS.values()), 'Bundled protein fixtures missing'
def pdb_to_ase(pdb_path, model_index=0):
    atoms = read(str(pdb_path), index=model_index)
    atoms.set_pbc(False)
    pos = atoms.get_positions()
    pad = 8.0
    size = pos.max(axis=0) - pos.min(axis=0) + 2*pad
    atoms.set_cell(np.diag(size))
    atoms.set_positions(pos - pos.min(axis=0) + pad)
    return atoms
calmodulin_apo = pdb_to_ase(PDB_PATHS[APO_PDB])
calmodulin_holo = pdb_to_ase(PDB_PATHS[HOLO_PDB])
print(f'Apo  ({APO_PDB}): {len(calmodulin_apo)} heavy atoms')
print(f'Holo ({HOLO_PDB}): {len(calmodulin_holo)} heavy atoms')

In [ ]:
cs_settings = ContactSpaceModel.parse_obj({
    'mode'     : 'ionic',
    'cutoff'   : 7,
    'alpha'    : 0.8,
    'spread'   : 0.5,
    'threshold': 0.7,
})

symfuncsettings = SymmetryFunctionsModel.parse_obj({'functions': [
    # Keep your existing baseline for direct comparison
    
    # New: larger radii with compositional and higher order
    SymFuncModel.parse_obj({
        'type': 'ac', 'radius': 10., 'order': 12,
        'compositional': True, 'structural': True, 'radial': True,
    }),
    SymFuncModel.parse_obj({
        'type': 'ac', 'radius': 12., 'order': 14,
        'compositional': True, 'structural': True, 'radial': True,
    }),
]})

symmetryfunctions = SymmetryFunctionsParser(symfuncsettings).parse()
symmetryfunctions = SymmetryFunctionsParser(symfuncsettings).parse()

def run_pipeline(atoms, label):
    print(f'\n=== {label} ===')
    grid   = Grid(cell=atoms.cell)
    system = System(grid, atoms, dimension=0, axis=1)
    print('  Generating contact space...')
    cs = ContactSpaceGenerator(cs_settings).generate(system)
    main_region = cs.data.groupby('region').size().idxmax()
    mask = cs.data['region'] == main_region
    cs.data = cs.data[mask].reset_index(drop=True)
    print(f'  {len(atoms)} atoms | {len(cs.data)} CS points')
    m = Maps(system, symmetryfunctions, cs)
    print('  Computing ACSF descriptors...')
    m.atcontactspace()
    m.reduce(3)
    print(f'  Done. {m.data.shape[0]} points × {len(m.features)} ACSF features')
    return atoms, cs, m

protein1, cs1, maps1 = run_pipeline(calmodulin_apo,  f'Apo ({APO_PDB})')
protein2, cs2, maps2 = run_pipeline(calmodulin_holo, f'Holo ({HOLO_PDB})')


---
## Section 5 — Add Physicochemical Descriptors

In [ ]:
# Cutoffs:
# Electrostatic: 8A — long-range Coulomb, needs wider radius
# Hydrophobicity: 5A — local environment, tighter radius
R_CUT_ELEC  = 8.0
R_CUT_HYDRO = 5.0

print('=== Apo ===')
maps1 = add_physicochemical_descriptors(
    maps1, calmodulin_apo, str(PDB_PATHS[APO_PDB]), model_index=0,
    r_cut_elec=R_CUT_ELEC, r_cut_hydro=R_CUT_HYDRO
)

print('\n=== Holo ===')
maps2 = add_physicochemical_descriptors(
    maps2, calmodulin_holo, str(PDB_PATHS[HOLO_PDB]), model_index=0,
    r_cut_elec=R_CUT_ELEC, r_cut_hydro=R_CUT_HYDRO
)


all_features = list(maps1.features) + ['electrostatic', 'hydrophobicity']
print(f'\nTotal features per CS point: {len(all_features)}')
print(f'Features: {all_features}')


In [ ]:
# Extract physicochemical from maps data into named arrays
elec_apo   = maps1.data['electrostatic'].values
hydro_apo  = maps1.data['hydrophobicity'].values
elec_holo  = maps2.data['electrostatic'].values
hydro_holo = maps2.data['hydrophobicity'].values

# Save everything so you don't lose it again
maps1.data.to_parquet('maps1_full.parquet')
maps2.data.to_parquet('maps2_full.parquet')
np.save('acsf_feature_names.npy', np.array(maps1.features))
print("Saved. Columns:", list(maps1.data.columns))

---
## Section 6 — Statistical Comparison

In [ ]:
rows = []
for f in all_features:
    v1 = maps1.data[f].values
    v2 = maps2.data[f].values
    ks_stat, ks_p  = ks_2samp(v1, v2)
    mw_stat, mw_p  = mannwhitneyu(v1, v2, alternative='two-sided')
    rows.append({
        'feature'   : f,
        'mean_apo'  : v1.mean(),
        'mean_holo' : v2.mean(),
        'delta_mean': v2.mean() - v1.mean(),
        'std_apo'   : v1.std(),
        'std_holo'  : v2.std(),
        'KS_stat'   : ks_stat,
        'KS_pvalue' : ks_p,
        'MW_pvalue' : mw_p,
    })

stats_df = pd.DataFrame(rows).set_index('feature')
print(f'=== Calmodulin Apo ({APO_PDB}) vs Holo ({HOLO_PDB}) ===')
print(stats_df.round(4).to_string())

# Highlight new descriptors
print('\n=== New physicochemical descriptors ===')
print(stats_df.loc[['electrostatic','hydrophobicity']].round(4).to_string())


---
## Section 7 — Visualization

In [ ]:
n_feats = len(all_features)
ncols   = min(4, n_feats)
nrows   = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.array(axes).flatten()

for i, f in enumerate(all_features):
    v1 = maps1.data[f].values
    v2 = maps2.data[f].values
    # Highlight the two new descriptors
    c1, c2 = ('steelblue', 'tomato') if f in ['electrostatic','hydrophobicity'] else ('#7EC8E3', '#FFB6A3')
    axes[i].hist(v1, bins=40, alpha=0.6, color=c1, label=f'Apo ({APO_PDB})',  density=True)
    axes[i].hist(v2, bins=40, alpha=0.6, color=c2, label=f'Holo ({HOLO_PDB})', density=True)
    ks = stats_df.loc[f, 'KS_stat']
    label = f'★ {f}' if f in ['electrostatic','hydrophobicity'] else f
    axes[i].set_title(f'{label}\nKS={ks:.3f}', fontsize=9,
                      fontweight='bold' if f in ['electrostatic','hydrophobicity'] else 'normal')
    axes[i].legend(fontsize=7)
    axes[i].set_xlabel('value')
    axes[i].set_ylabel('density')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Calmodulin Apo vs Holo — ACSF + Electrostatics + Hydrophobicity',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
def plot_descriptor_3d(atoms, maps, feature, title, colorscale='RdBu'):
    """3D view of a single descriptor on the contact space."""
    fig = go.Figure()
    pos = atoms.get_positions()
    sym = np.array(atoms.get_chemical_symbols())
    for elem in sorted(set(sym)):
        mask = sym == elem
        p = pos[mask]
        fig.add_trace(go.Scatter3d(
            x=p[:,0], y=p[:,1], z=p[:,2],
            mode='markers', name=elem,
            marker=dict(size=3, color=_atom_color(elem), opacity=0.4),
        ))
    cvals = maps.data[feature].values
    fig.add_trace(go.Scatter3d(
        x=maps.data['x'], y=maps.data['y'], z=maps.data['z'],
        mode='markers', name=feature,
        marker=dict(size=5, color=cvals, colorscale=colorscale,
                    opacity=0.9,
                    colorbar=dict(title=feature, thickness=14),
                    showscale=True),
        showlegend=False,
    ))
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data',
                   xaxis_title='x (Å)', yaxis_title='y (Å)', zaxis_title='z (Å)'),
        width=950, height=720, template='plotly_white',
    )
    return fig

# Electrostatic potential — red/blue diverging colorscale
plot_descriptor_3d(protein1, maps1, 'electrostatic',
    f'Apo ({APO_PDB}) — Electrostatic Potential', colorscale='RdBu').show()
plot_descriptor_3d(protein2, maps2, 'electrostatic',
    f'Holo ({HOLO_PDB}) — Electrostatic Potential', colorscale='RdBu').show()

# Hydrophobicity — green/orange
plot_descriptor_3d(protein1, maps1, 'hydrophobicity',
    f'Apo ({APO_PDB}) — Hydrophobicity', colorscale='RdYlGn').show()
plot_descriptor_3d(protein2, maps2, 'hydrophobicity',
    f'Holo ({HOLO_PDB}) — Hydrophobicity', colorscale='RdYlGn').show()


In [ ]:
def plot_side_by_side(atoms1, maps1, atoms2, maps2, feature, title, colorscale='RdBu'):
    """Side-by-side apo/holo comparison for one descriptor."""
    v1, v2 = maps1.data[feature].values, maps2.data[feature].values
    cmin   = min(v1.min(), v2.min())
    cmax   = max(v1.max(), v2.max())

    fig = make_subplots(rows=1, cols=2,
        specs=[[{'type':'scene'}, {'type':'scene'}]],
        subplot_titles=[f'Apo ({APO_PDB})', f'Holo ({HOLO_PDB})'],
        horizontal_spacing=0.02)

    for col, (atoms, maps, vals) in enumerate(
            [(atoms1, maps1, v1), (atoms2, maps2, v2)], start=1):
        pos = atoms.get_positions()
        sym = np.array(atoms.get_chemical_symbols())
        for elem in sorted(set(sym)):
            mask = sym == elem
            p = pos[mask]
            fig.add_trace(go.Scatter3d(
                x=p[:,0], y=p[:,1], z=p[:,2], mode='markers',
                name=elem, showlegend=(col==1),
                marker=dict(size=3, color=_atom_color(elem), opacity=0.3),
            ), row=1, col=col)

        fig.add_trace(go.Scatter3d(
            x=maps.data['x'], y=maps.data['y'], z=maps.data['z'],
            mode='markers', showlegend=False,
            marker=dict(
                size=5, color=vals, colorscale=colorscale,
                cmin=cmin, cmax=cmax, opacity=0.9,
                showscale=(col==2),
                colorbar=dict(title=feature, thickness=14, x=1.02) if col==2 else None,
            ),
        ), row=1, col=col)

    scene = dict(aspectmode='data',
                 xaxis_title='x', yaxis_title='y', zaxis_title='z')
    fig.update_layout(title_text=title, width=1400, height=680,
                      template='plotly_white', scene=scene, scene2=scene)
    return fig


plot_side_by_side(protein1, maps1, protein2, maps2, 'electrostatic',
    'Electrostatic Potential — Apo vs Holo', colorscale='RdBu').show()

plot_side_by_side(protein1, maps1, protein2, maps2, 'hydrophobicity',
    'Hydrophobicity — Apo vs Holo', colorscale='RdYlGn').show()


---
## Section 8 — Combined Feature Matrix

In [ ]:
# Build combined feature matrix: ACSF + electrostatic + hydrophobicity
# Useful for downstream ML or clustering

def get_feature_matrix(maps_obj, features=None):
    """Return (N_cs, N_features) array and feature names."""
    if features is None:
        features = list(maps_obj.features) + ['electrostatic', 'hydrophobicity']
    X = maps_obj.data[features].values
    return X, features

X1, feat_names = get_feature_matrix(maps1)
X2, _          = get_feature_matrix(maps2)

print(f'Apo  feature matrix: {X1.shape}')
print(f'Holo feature matrix: {X2.shape}')
print(f'Features: {feat_names}')

# Quick correlation check: do new features correlate with ACSF features?
import numpy as np
print('\nCorrelation of electrostatic with ACSF features (apo):')
for f in maps1.features:
    r = np.corrcoef(maps1.data[f].values, maps1.data['electrostatic'].values)[0,1]
    print(f'  {f}: {r:.3f}')
print('\nCorrelation of hydrophobicity with ACSF features (apo):')
for f in maps1.features:
    r = np.corrcoef(maps1.data[f].values, maps1.data['hydrophobicity'].values)[0,1]
    print(f'  {f}: {r:.3f}')


DIMERED CLASSIFICATION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import (classification_report, roc_auc_score,
                              RocCurveDisplay, roc_curve)
from sklearn.inspection import permutation_importance
import umap

import warnings
# Keep warnings visible during verification.

# ── Colour palette ─────────────────────────────────────────────────────
APO_COLOR  = '#4C9BE8'   # blue
HOLO_COLOR = '#E8714C'   # orange
APO_LABEL  = 'Apo (1CFD)'
HOLO_LABEL = 'Holo (1CLL)'

print('All imports OK ✓')


In [ ]:
# ── Build combined feature matrix ──────────────────────────────────────
# Assumes maps1 and maps2 already exist from the extended descriptors notebook.
# All 54 features: 52 ACSF + electrostatic + hydrophobicity

ALL_FEATURES = list(maps1.features) + ['electrostatic', 'hydrophobicity']

X_apo  = maps1.data[ALL_FEATURES].values.astype(np.float64)   # (N_apo, 54)
X_holo = maps2.data[ALL_FEATURES].values.astype(np.float64)   # (N_holo, 54)

X = np.vstack([X_apo, X_holo])                                 # (N_total, 54)
y = np.array([0]*len(X_apo) + [1]*len(X_holo))                # 0=apo, 1=holo
labels = np.where(y == 0, APO_LABEL, HOLO_LABEL)

print(f'Apo  CS points : {len(X_apo):,}')
print(f'Holo CS points : {len(X_holo):,}')
print(f'Total          : {len(X):,}')
print(f'Features       : {len(ALL_FEATURES)}')
print(f'Class balance  : apo={len(X_apo)/len(X)*100:.1f}%  holo={len(X_holo)/len(X)*100:.1f}%')

# ── Standardise — zero mean, unit variance ─────────────────────────────
# Essential before PCA/UMAP — features have very different scales
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

print(f'\nAfter scaling — mean: {X_sc.mean():.6f}  std: {X_sc.std():.6f}')

# ── Stratified subsample for expensive operations (UMAP, silhouette) ───
# Uses equal numbers from each class to avoid bias
N_SAMPLE = 5000   # per class — adjust down if slow, up for more precision
rng = np.random.default_rng(42)
idx_apo  = rng.choice(np.where(y==0)[0], size=min(N_SAMPLE, len(X_apo)),  replace=False)
idx_holo = rng.choice(np.where(y==1)[0], size=min(N_SAMPLE, len(X_holo)), replace=False)
idx_sub  = np.sort(np.concatenate([idx_apo, idx_holo]))

X_sub    = X_sc[idx_sub]
y_sub    = y[idx_sub]
lab_sub  = labels[idx_sub]

print(f'\nSubsample: {len(X_sub):,} points ({N_SAMPLE} per class)')


---
## Section 2 — PCA

In [ ]:
# ── Fit PCA on full scaled matrix ──────────────────────────────────────
pca_full = PCA(n_components=min(X_sc.shape[1], len(X_sc)))
pca_full.fit(X_sc)

evr     = pca_full.explained_variance_ratio_
evr_cum = np.cumsum(evr)

# How many components for 90% / 95% variance?
n90 = np.searchsorted(evr_cum, 0.90) + 1
n95 = np.searchsorted(evr_cum, 0.95) + 1

print(f'Components for 90% variance: {n90}')
print(f'Components for 95% variance: {n95}')
print(f'PC1 explains: {evr[0]*100:.1f}%')
print(f'PC2 explains: {evr[1]*100:.1f}%')
print(f'PC1+PC2:      {evr_cum[1]*100:.1f}%')
print(f'PC1-PC5:      {evr_cum[4]*100:.1f}%')
print(f'PC1-PC10:     {evr_cum[9]*100:.1f}%')


In [ ]:
# ── Scree plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Individual variance
axes[0].bar(range(1, 21), evr[:20]*100, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot — top 20 PCs')
axes[0].axvline(n90, color='red',    ls='--', lw=1.5, label=f'90% ({n90} PCs)')
axes[0].axvline(n95, color='orange', ls='--', lw=1.5, label=f'95% ({n95} PCs)')
axes[0].legend()

# Cumulative
axes[1].plot(range(1, len(evr_cum)+1), evr_cum*100, 'o-', color='steelblue',
             ms=3, lw=1.5)
axes[1].axhline(90, color='red',    ls='--', lw=1.5, label='90%')
axes[1].axhline(95, color='orange', ls='--', lw=1.5, label='95%')
axes[1].axvline(n90, color='red',    ls='--', lw=1.0, alpha=0.5)
axes[1].axvline(n95, color='orange', ls='--', lw=1.0, alpha=0.5)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Variance')
axes[1].legend()

plt.suptitle('PCA — Variance Explained', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Project subsample onto PC1-PC4 ────────────────────────────────────
pca10    = PCA(n_components=10)
X_pca    = pca10.fit_transform(X_sc)       # full dataset
X_pca_s  = X_pca[idx_sub]                  # subsample for plotting

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
pairs = [(0,1), (0,2), (1,2), (0,3), (1,3), (2,3)]

for ax, (i, j) in zip(axes.flatten(), pairs):
    for cls, color, label in [(0, APO_COLOR, APO_LABEL), (1, HOLO_COLOR, HOLO_LABEL)]:
        mask = y_sub == cls
        ax.scatter(X_pca_s[mask, i], X_pca_s[mask, j],
                   c=color, alpha=0.15, s=4, label=label, rasterized=True)
    ax.set_xlabel(f'PC{i+1} ({pca10.explained_variance_ratio_[i]*100:.1f}%)')
    ax.set_ylabel(f'PC{j+1} ({pca10.explained_variance_ratio_[j]*100:.1f}%)')
    ax.legend(markerscale=4, fontsize=8)
    ax.set_title(f'PC{i+1} vs PC{j+1}')

plt.suptitle('PCA Projections — Apo vs Holo CS Points', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── PC loadings — which features drive each PC ────────────────────────
components = pca10.components_   # (10, 54)
loadings_df = pd.DataFrame(
    components[:5].T,
    index=ALL_FEATURES,
    columns=[f'PC{i+1}' for i in range(5)]
)

# Top 10 contributors to PC1 and PC2
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, pc in zip(axes, ['PC1', 'PC2']):
    top = loadings_df[pc].abs().nlargest(15).index
    vals = loadings_df.loc[top, pc].sort_values()
    colors = [HOLO_COLOR if v > 0 else APO_COLOR for v in vals]
    vals.plot.barh(ax=ax, color=colors, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{pc} — top 15 loadings\n'
                 f'({pca10.explained_variance_ratio_[int(pc[2])-1]*100:.1f}% variance)')
    ax.set_xlabel('Loading')

plt.suptitle('PCA Loadings — Feature Contributions', fontsize=13)
plt.tight_layout()
plt.show()

# Print top features for each PC
print('=== Top 5 features per PC ===')
for pc in [f'PC{i+1}' for i in range(5)]:
    top5 = loadings_df[pc].abs().nlargest(5).index.tolist()
    print(f'{pc}: {top5}')


In [ ]:
# ── Silhouette score — how well separated are the classes in PCA space ─
from sklearn.metrics import silhouette_score

sil_scores = {}
for n_comp in [2, 3, 5, 10]:
    X_proj = X_pca[idx_sub, :n_comp]
    sil = silhouette_score(X_proj, y_sub, sample_size=min(5000, len(y_sub)),
                           random_state=42)
    sil_scores[n_comp] = sil
    print(f'Silhouette (PCA {n_comp}D): {sil:.4f}')

# Silhouette interpretation:
# > 0.5  strong structure
# 0.25-0.5  medium structure
# < 0.25  weak or no structure
print()
print('Interpretation: >0.5 strong | 0.25-0.5 medium | <0.25 weak')


---
## Section 3 — UMAP

Non-linear embedding that preserves local and global structure. Uses the balanced subsample. Expect ~2-5 minutes.

In [ ]:
# ── UMAP embedding ─────────────────────────────────────────────────────
# n_neighbors: local structure scale (15=local, 50=global)
# min_dist: how tightly points cluster (0.05=tight, 0.5=spread)
print('Fitting UMAP... (this takes 2-5 minutes for 10k points)')

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=30,       # larger = more global structure
    min_dist=0.1,
    metric='euclidean',
    random_state=42,
    verbose=False,
)
X_umap = reducer.fit_transform(X_sub)

print(f'UMAP done. Embedding shape: {X_umap.shape}')


In [ ]:
# ── UMAP scatter — matplotlib ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: coloured by class
for cls, color, label in [(0, APO_COLOR, APO_LABEL), (1, HOLO_COLOR, HOLO_LABEL)]:
    mask = y_sub == cls
    axes[0].scatter(X_umap[mask, 0], X_umap[mask, 1],
                    c=color, alpha=0.2, s=3, label=label, rasterized=True)
axes[0].set_title('UMAP — Apo vs Holo', fontsize=12)
axes[0].legend(markerscale=5, fontsize=10)
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# Right: coloured by electrostatic potential
elec_sub = maps1.data['electrostatic'].values[idx_apo[:N_SAMPLE]]
elec_sub = np.concatenate([
    maps1.data['electrostatic'].values[idx_apo],
    maps2.data['electrostatic'].values[idx_holo - len(X_apo)]
                            if idx_holo.max() >= len(X_apo)
                            else maps2.data['electrostatic'].values[idx_holo]
])
# Safe indexing for electrostatic values in subsample
elec_vals = np.concatenate([
    maps1.data['electrostatic'].values[idx_apo - 0],
    maps2.data['electrostatic'].values[idx_holo - len(X_apo)]
])

sc = axes[1].scatter(X_umap[:, 0], X_umap[:, 1],
                     c=elec_vals, cmap='RdBu_r',
                     alpha=0.3, s=3, rasterized=True,
                     vmin=np.percentile(elec_vals, 5),
                     vmax=np.percentile(elec_vals, 95))
plt.colorbar(sc, ax=axes[1], label='Electrostatic potential')
axes[1].set_title('UMAP coloured by Electrostatic Potential', fontsize=12)
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

plt.suptitle('UMAP Embedding of MapSy Feature Space', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── UMAP coloured by hydrophobicity ───────────────────────────────────
hydro_vals = np.concatenate([
    maps1.data['hydrophobicity'].values[idx_apo],
    maps2.data['hydrophobicity'].values[idx_holo - len(X_apo)]
])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Hydrophobicity
sc1 = axes[0].scatter(X_umap[:, 0], X_umap[:, 1],
                      c=hydro_vals, cmap='RdYlGn',
                      alpha=0.3, s=3, rasterized=True,
                      vmin=np.percentile(hydro_vals, 5),
                      vmax=np.percentile(hydro_vals, 95))
plt.colorbar(sc1, ax=axes[0], label='Hydrophobicity (KD)')
axes[0].set_title('UMAP coloured by Hydrophobicity', fontsize=12)
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# PC1 value
pc1_vals = X_pca[idx_sub, 0]
sc2 = axes[1].scatter(X_umap[:, 0], X_umap[:, 1],
                      c=pc1_vals, cmap='plasma',
                      alpha=0.3, s=3, rasterized=True,
                      vmin=np.percentile(pc1_vals, 5),
                      vmax=np.percentile(pc1_vals, 95))
plt.colorbar(sc2, ax=axes[1], label='PC1 score')
axes[1].set_title('UMAP coloured by PC1 score', fontsize=12)
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

plt.suptitle('UMAP — Physicochemical Overlay', fontsize=13)
plt.tight_layout()
plt.show()


---
## Section 4 — LDA

Finds the single linear axis that maximally separates apo from holo. The most interpretable view of class separation.

In [ ]:
# ── Linear Discriminant Analysis ──────────────────────────────────────
# LDA finds the linear combination of features that maximally separates classes.
# For 2 classes there is exactly 1 discriminant axis (LD1).
lda = LinearDiscriminantAnalysis()
lda.fit(X_sc, y)

X_ld1 = lda.transform(X_sc).ravel()   # project all points onto LD1

# LDA score (leave-one-out cross-val approximation via full fit)
ld1_apo  = X_ld1[y == 0]
ld1_holo = X_ld1[y == 1]

print(f'LD1 mean apo:  {ld1_apo.mean():.3f} ± {ld1_apo.std():.3f}')
print(f'LD1 mean holo: {ld1_holo.mean():.3f} ± {ld1_holo.std():.3f}')
print(f'Separation (Cohen d): {abs(ld1_apo.mean()-ld1_holo.mean()) / np.sqrt((ld1_apo.std()**2+ld1_holo.std()**2)/2):.3f}')

# ── LDA coefficient loadings ───────────────────────────────────────────
lda_coef = pd.Series(lda.coef_[0], index=ALL_FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# LD1 distribution
axes[0].hist(ld1_apo,  bins=80, alpha=0.6, color=APO_COLOR,  density=True, label=APO_LABEL)
axes[0].hist(ld1_holo, bins=80, alpha=0.6, color=HOLO_COLOR, density=True, label=HOLO_LABEL)
axes[0].axvline(ld1_apo.mean(),  color=APO_COLOR,  ls='--', lw=2)
axes[0].axvline(ld1_holo.mean(), color=HOLO_COLOR, ls='--', lw=2)
axes[0].set_xlabel('LD1 Score')
axes[0].set_ylabel('Density')
axes[0].set_title('LDA Discriminant Score Distribution')
axes[0].legend()

# Top loadings
top_lda = pd.concat([lda_coef.head(12), lda_coef.tail(12)])
colors  = [APO_COLOR if v < 0 else HOLO_COLOR for v in top_lda]
top_lda.plot.barh(ax=axes[1], color=colors, edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('LDA Coefficients — Top 24 Features\n'
                  f'(blue=drives apo | orange=drives holo)')
axes[1].set_xlabel('LDA Coefficient')

plt.suptitle('Linear Discriminant Analysis', fontsize=13)
plt.tight_layout()
plt.show()

# Print top 10 features driving separation
print('\n=== Top 10 features pointing toward HOLO (positive LD1) ===')
print(lda_coef.tail(10).to_string())
print('\n=== Top 10 features pointing toward APO (negative LD1) ===')
print(lda_coef.head(10).to_string())


---
## Section 5 — Classification

Can we predict whether a CS point belongs to apo or holo from its descriptor vector alone? Logistic regression (linear) and random forest (non-linear), both with 5-fold CV.

In [ ]:
# ── Stratified 5-fold CV ───────────────────────────────────────────────
# Use subsample to keep runtime manageable (~2 min)
print('Running 5-fold cross-validation...')
print('(using balanced subsample of 10,000 points)')

from sklearn.pipeline import make_pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Logistic Regression ────────────────────────────────────────────────
lr = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
lr_results = cross_validate(
    make_pipeline(StandardScaler(), lr), X[idx_sub], y_sub, cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1'],
    return_train_score=True,
)

print('\n=== Logistic Regression ===')
for metric in ['accuracy', 'roc_auc', 'f1']:
    test  = lr_results[f'test_{metric}']
    train = lr_results[f'train_{metric}']
    print(f'  {metric:12s}: test={test.mean():.3f}±{test.std():.3f}  '
          f'train={train.mean():.3f}±{train.std():.3f}')

# ── Random Forest ──────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             n_jobs=4, random_state=42)
rf_results = cross_validate(
    make_pipeline(StandardScaler(), rf), X[idx_sub], y_sub, cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1'],
    return_train_score=True,
)

print('\n=== Random Forest ===')
for metric in ['accuracy', 'roc_auc', 'f1']:
    test  = rf_results[f'test_{metric}']
    train = rf_results[f'train_{metric}']
    print(f'  {metric:12s}: test={test.mean():.3f}±{test.std():.3f}  '
          f'train={train.mean():.3f}±{train.std():.3f}')


In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (model, name, color) in zip(axes, [
        (LogisticRegression(max_iter=2000, C=1.0, random_state=42),
         'Logistic Regression', 'steelblue'),
        (RandomForestClassifier(n_estimators=200, n_jobs=4, random_state=42),
         'Random Forest', 'darkorange'),
]):
    tprs, aucs = [], []
    mean_fpr   = np.linspace(0, 1, 100)

    for fold, (train_idx, test_idx) in enumerate(cv.split(X_sub, y_sub)):
        fold_model = make_pipeline(StandardScaler(), model)
        fold_model.fit(X[idx_sub][train_idx], y_sub[train_idx])
        proba = fold_model.predict_proba(X[idx_sub][test_idx])[:, 1]
        fpr, tpr, _ = roc_curve(y_sub[test_idx], proba)
        auc = roc_auc_score(y_sub[test_idx], proba)
        aucs.append(auc)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
        ax.plot(fpr, tpr, alpha=0.2, color=color, lw=1)

    mean_tpr = np.mean(tprs, axis=0)
    std_tpr  = np.std(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, color=color, lw=2.5,
            label=f'Mean AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}')
    ax.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr,
                    alpha=0.15, color=color)
    ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC=0.5)')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC — {name}')
    ax.legend(fontsize=10)

plt.suptitle('Classification Performance — Apo vs Holo from Surface Descriptors', fontsize=13)
plt.tight_layout()
plt.show()


---
## Section 6 — Feature Importance

Which of the 54 features contribute most to distinguishing apo from holo? Three complementary approaches: RF built-in importance, permutation importance, LR coefficients.

In [ ]:
# ── Train final models on full subsample ──────────────────────────────
lr_final = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
lr_final.fit(X_sub, y_sub)

rf_final = RandomForestClassifier(n_estimators=300, n_jobs=4, random_state=42)
rf_final.fit(X_sub, y_sub)

# ── Random Forest built-in importance ─────────────────────────────────
rf_imp = pd.Series(rf_final.feature_importances_, index=ALL_FEATURES)

# ── Logistic Regression coefficients (abs = importance) ───────────────
lr_imp = pd.Series(np.abs(lr_final.coef_[0]), index=ALL_FEATURES)

# ── Permutation importance on RF (more reliable than built-in) ─────────
print('Computing permutation importance (takes ~1 min)...')
perm = permutation_importance(
    rf_final, X_sub, y_sub,
    n_repeats=10, random_state=42, n_jobs=4, scoring='roc_auc'
)
perm_imp = pd.Series(perm.importances_mean, index=ALL_FEATURES)
perm_std = pd.Series(perm.importances_std,  index=ALL_FEATURES)

# ── Combined ranking ───────────────────────────────────────────────────
# Rank each feature by each method, average ranks
rf_rank   = rf_imp.rank(ascending=False)
lr_rank   = lr_imp.rank(ascending=False)
perm_rank = perm_imp.rank(ascending=False)
avg_rank  = (rf_rank + lr_rank + perm_rank) / 3

importance_df = pd.DataFrame({
    'RF_importance'  : rf_imp,
    'LR_coef_abs'    : lr_imp,
    'Perm_importance': perm_imp,
    'Perm_std'       : perm_std,
    'Avg_rank'       : avg_rank,
}).sort_values('Avg_rank')

print('\n=== Top 20 features by average rank across all methods ===')
print(importance_df.head(20).round(4).to_string())


In [ ]:
# ── Feature importance visualisation ──────────────────────────────────
top_n = 20
top_features = importance_df.head(top_n).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Colour by feature type
def feat_color(f):
    if f == 'electrostatic':  return '#E63946'
    if f == 'hydrophobicity': return '#2A9D8F'
    if 'RC' in f:             return '#E9C46A'
    return '#A8DADC'   # RS

# RF importance
rf_top = rf_imp[top_features].sort_values()
colors = [feat_color(f) for f in rf_top.index]
rf_top.plot.barh(ax=axes[0], color=colors, edgecolor='white', alpha=0.9)
axes[0].set_title('Random Forest\nBuilt-in Importance')
axes[0].set_xlabel('Importance')

# Permutation importance
perm_top = perm_imp[top_features].sort_values()
perm_s   = perm_std[top_features].reindex(perm_top.index)
colors   = [feat_color(f) for f in perm_top.index]
perm_top.plot.barh(ax=axes[1], color=colors, edgecolor='white', alpha=0.9,
                   xerr=perm_s.values)
axes[1].set_title('Permutation Importance\n(RF model, AUC metric)')
axes[1].set_xlabel('Mean AUC decrease')

# LR coefficients
lr_top = lr_imp[top_features].sort_values()
colors = [feat_color(f) for f in lr_top.index]
lr_top.plot.barh(ax=axes[2], color=colors, edgecolor='white', alpha=0.9)
axes[2].set_title('Logistic Regression\n|Coefficient|')
axes[2].set_xlabel('|Coefficient|')

# Legend
legend_patches = [
    Patch(color='#E63946', label='Electrostatic'),
    Patch(color='#2A9D8F', label='Hydrophobicity'),
    Patch(color='#E9C46A', label='ACSF RC (compositional)'),
    Patch(color='#A8DADC', label='ACSF RS (structural)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.suptitle(f'Top {top_n} Feature Importances — Apo vs Holo Classification',
             fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Final summary table ────────────────────────────────────────────────
print('=== SUMMARY ===')
print()
print(f'Dataset: {len(X_apo):,} apo + {len(X_holo):,} holo CS points, 54 features')
print()
print('Classification (5-fold CV, balanced subsample):')
print(f'  Logistic Regression AUC: {np.mean(lr_results["test_roc_auc"]):.3f} '
      f'± {np.std(lr_results["test_roc_auc"]):.3f}')
print(f'  Random Forest       AUC: {np.mean(rf_results["test_roc_auc"]):.3f} '
      f'± {np.std(rf_results["test_roc_auc"]):.3f}')
print()
print(f'LDA separation (Cohen d): '
      f'{abs(ld1_apo.mean()-ld1_holo.mean()) / np.sqrt((ld1_apo.std()**2+ld1_holo.std()**2)/2):.3f}')
print()
print('Top 5 features by average rank:')
for i, (feat, row) in enumerate(importance_df.head(5).iterrows(), 1):
    ftype = ('Electrostatic' if feat=='electrostatic'
             else 'Hydrophobicity' if feat=='hydrophobicity'
             else 'RC (compositional)' if 'RC' in feat
             else 'RS (structural)')
    print(f'  {i}. {feat:35s} [{ftype}]  avg_rank={row.Avg_rank:.1f}')
print()
print('New physicochemical descriptors in top 20:')
for f in importance_df.head(20).index:
    if f in ['electrostatic', 'hydrophobicity']:
        rank = importance_df.index.get_loc(f) + 1
        print(f'  {f}: rank {rank}/54')


In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
import copy

# ── 1. Load structures ────────────────────────────────────────────────────────
from Bio.PDB import PDBParser, PDBIO, Select

class ProteinOnly(Select):
    def accept_residue(self, r):
        return 1 if r.get_id()[0] == ' ' else 0
    def accept_atom(self, a):
        return 1 if a.element != 'H' else 0

# ── 2. Load MapSy objects from saved parquet files ───────────────────────────
# (saved earlier with maps1.data.to_parquet / maps2.data.to_parquet)
print("Loading saved descriptor data...")

class SimpleContainer:
    pass

maps1 = SimpleContainer()
maps1.data     = pd.read_parquet('maps1_full.parquet')
maps1.features = list(np.load('acsf_feature_names.npy', allow_pickle=True))

maps2 = SimpleContainer()
maps2.data     = pd.read_parquet('maps2_full.parquet')
maps2.features = maps1.features

print(f"Apo  loaded: {maps1.data.shape}")
print(f"Holo loaded: {maps2.data.shape}")
print(f"Features: {len(maps1.features)}")

# ── 3. Add physicochemical descriptors ───────────────────────────────────────
# If you saved them separately, load here. Otherwise recompute:
# Check if already in data
if 'electrostatic' not in maps1.data.columns:
    print("Physicochemical descriptors not found in parquet — need to recompute.")
    print("Load your ASE atoms objects and run compute_electrostatic / compute_hydrophobicity")
    print("Or save maps1.data (with all columns) to parquet after computing them.")
else:
    print("Physicochemical descriptors found in parquet.")

all_features = maps1.features + ['electrostatic', 'hydrophobicity']
print(f"Total features available: {len(all_features)}")

# ── 4. Subsample for clustering ───────────────────────────────────────────────
print("\nSubsampling for clustering...")
rng = np.random.default_rng(42)

n_sub = 5000
idx_apo_sub  = rng.choice(len(maps1.data), n_sub, replace=False)
idx_holo_sub = rng.choice(len(maps2.data), n_sub, replace=False)

data_apo_sub  = maps1.data.iloc[idx_apo_sub].reset_index(drop=True)
data_holo_sub = maps2.data.iloc[idx_holo_sub].reset_index(drop=True)

print(f"Subsampled: {len(data_apo_sub)} apo, {len(data_holo_sub)} holo points")

# ── 5. Scale features ─────────────────────────────────────────────────────────
X_apo_sub  = data_apo_sub[maps1.features].values
X_holo_sub = data_holo_sub[maps1.features].values

scaler     = StandardScaler()
X_apo_sc   = scaler.fit_transform(X_apo_sub)
X_holo_sc  = scaler.transform(X_holo_sub)

# Add scaled features back to dataframes for MapSy clustering
feat_names_sc = [f + '_sc' for f in maps1.features]
data_apo_sub[feat_names_sc]  = X_apo_sc
data_holo_sub[feat_names_sc] = X_holo_sc

print("Features scaled.")

# ── 6. Now load actual MapSy objects to run cluster() ─────────────────────────
# You need real MapSy Maps objects for cluster() and scatter()
# Run the contact space + ACSF computation first, then cluster on subsampled data
print("\nTo run MapSy clustering, you need the original Maps objects.")
print("Run your pipeline notebook first to get maps1 and maps2,")
print("then run the clustering block below.")
print("\nEstimated RAM for spectral clustering on 5000 points:")
print(f"  Affinity matrix: {5000**2 * 8 / 1e9:.2f} GB — manageable")
print(f"  Full 37k points: {37000**2 * 8 / 1e9:.1f} GB — crashes")

In [ ]:
print(maps1.data.columns.tolist())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ── Setup ─────────────────────────────────────────────────────────────────────
n_sub = 5000
rng   = np.random.default_rng(42)

acsf_features = maps1.features

idx_apo  = rng.choice(len(maps1.data), n_sub, replace=False)
idx_holo = rng.choice(len(maps2.data), n_sub, replace=False)

data_apo  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo = maps2.data.iloc[idx_holo].reset_index(drop=True)

scaler    = StandardScaler()
X_apo_sc  = scaler.fit_transform(data_apo[acsf_features].values)
X_holo_sc = scaler.transform(data_holo[acsf_features].values)

# ── Sweep k from 2 to 25 ─────────────────────────────────────────────────────
k_range = range(2, 26)
results = {'apo': [], 'holo': []}

for k in k_range:
    print(f"k={k:2d} ...", end=' ')
    row_apo  = {'k': k}
    row_holo = {'k': k}

    for tag, X_sc, row in [('apo', X_apo_sc, row_apo),
                            ('holo', X_holo_sc, row_holo)]:
        sc     = SpectralClustering(n_clusters=k, n_init=5,
                                    affinity='nearest_neighbors',
                                    n_neighbors=15, random_state=42,
                                    n_jobs=4)
        labels = sc.fit_predict(X_sc)
        row['labels'] = labels
        row['sil']    = silhouette_score(X_sc, labels,
                                         sample_size=2000, random_state=42)
        row['db']     = davies_bouldin_score(X_sc, labels)
        print(f"{tag} sil={row['sil']:.4f} db={row['db']:.4f}", end='  ')

    results['apo'].append(row_apo)
    results['holo'].append(row_holo)
    print()

# ── Find optima ───────────────────────────────────────────────────────────────
for tag in ['apo', 'holo']:
    sils = [r['sil'] for r in results[tag]]
    dbs  = [r['db']  for r in results[tag]]
    ks   = [r['k']   for r in results[tag]]

    best_sil_k = ks[np.argmax(sils)]
    best_db_k  = ks[np.argmin(dbs)]
    print(f"\n{tag.upper()}:")
    print(f"  Best Silhouette: k={best_sil_k} ({max(sils):.4f})")
    print(f"  Best DB:         k={best_db_k}  ({min(dbs):.4f})")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, tag in enumerate(['apo', 'holo']):
    ks   = [r['k']   for r in results[tag]]
    sils = [r['sil'] for r in results[tag]]
    dbs  = [r['db']  for r in results[tag]]

    best_sil_k = ks[np.argmax(sils)]
    best_db_k  = ks[np.argmin(dbs)]

    # Silhouette
    ax = axes[0, col]
    ax.plot(ks, sils, 'o-', color='steelblue', linewidth=2)
    ax.axvline(best_sil_k, color='steelblue', linestyle='--', alpha=0.7,
               label=f'Best k={best_sil_k}')
    ax.set_xlabel('Number of clusters k')
    ax.set_ylabel('Silhouette score (higher = better)')
    ax.set_title(f'{tag.upper()} — Silhouette vs k')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Davies-Bouldin
    ax = axes[1, col]
    ax.plot(ks, dbs, 'o-', color='darkorange', linewidth=2)
    ax.axvline(best_db_k, color='darkorange', linestyle='--', alpha=0.7,
               label=f'Best k={best_db_k}')
    ax.set_xlabel('Number of clusters k')
    ax.set_ylabel('Davies-Bouldin score (lower = better)')
    ax.set_title(f'{tag.upper()} — Davies-Bouldin vs k')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Optimal Number of Surface Environment Clusters — Calmodulin\n'
             f'Spectral clustering, n={n_sub} subsampled CS points per structure',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('optimal_k_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# ── Extract best k labels from already-computed results ───────────────────────
best_configs = {
    'apo':  {'sil': 3,  'db': 6},
    'holo': {'sil': 4,  'db': 11},
}

# Rebuild subsampled data and PCA coords using same seed
rng       = np.random.default_rng(42)
n_sub     = 5000
idx_apo   = rng.choice(len(maps1.data), n_sub, replace=False)
idx_holo  = rng.choice(len(maps2.data), n_sub, replace=False)
data_apo  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo = maps2.data.iloc[idx_holo].reset_index(drop=True)
pca_apo   = data_apo[['pca0', 'pca1']].values
pca_holo  = data_holo[['pca0', 'pca1']].values

# Pull labels from results dict (already computed in sweep)
def get_labels(results_list, k):
    return next(r['labels'] for r in results_list if r['k'] == k)

labels = {
    ('apo',  'sil'): get_labels(results['apo'],  best_configs['apo']['sil']),
    ('apo',  'db'):  get_labels(results['apo'],  best_configs['apo']['db']),
    ('holo', 'sil'): get_labels(results['holo'], best_configs['holo']['sil']),
    ('holo', 'db'):  get_labels(results['holo'], best_configs['holo']['db']),
}

pca_coords = {'apo': pca_apo, 'holo': pca_holo}

# ── Plot helper ───────────────────────────────────────────────────────────────
def plot_cluster_panel(ax, pca_xy, cluster_labels, title, metric_str):
    k          = len(np.unique(cluster_labels))
    colors     = plt.cm.tab20(np.linspace(0, 1, k))
    centroids  = np.array([pca_xy[cluster_labels == i].mean(axis=0)
                            for i in range(k)])

    # Scatter
    for i in range(k):
        mask = cluster_labels == i
        ax.scatter(pca_xy[mask, 0], pca_xy[mask, 1],
                   color=colors[i], alpha=0.15, s=8, rasterized=True)

    # Adjacency: connect clusters proportional to shared boundary points
    adj = np.zeros((k, k))
    for i in range(k):
        for j in range(i + 1, k):
            w = np.sum((cluster_labels == i) | (cluster_labels == j))
            adj[i, j] = adj[j, i] = w

    G       = nx.from_numpy_array(adj, create_using=nx.Graph)
    pos     = {i: centroids[i] for i in range(k)}
    weights = [adj[u, v] / 2000 for u, v in G.edges()]

    nx.draw_networkx_edges(G, pos, width=weights, ax=ax,
                           alpha=0.45, edge_color='k')
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=100,
                           node_color=[colors[i] for i in range(k)],
                           edgecolors='k', linewidths=0.8)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7,
                            font_color='white', font_weight='bold')

    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2', fontsize=10)
    ax.set_title(f'{title}\n{metric_str}', fontsize=10, fontweight='bold')
    ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
    ax.set_aspect('equal')

# ── Figure 1: Best Silhouette ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, tag, pca_xy in [(axes[0], 'apo',  pca_apo),
                         (axes[1], 'holo', pca_holo)]:
    k      = best_configs[tag]['sil']
    lbl    = labels[(tag, 'sil')]
    r      = next(r for r in results[tag] if r['k'] == k)
    mstr   = f'k={k} | Silhouette={r["sil"]:.4f} | DB={r["db"]:.4f}'
    title  = f'{"Apo (1CFD)" if tag=="apo" else "Holo (1CLL)"} — Best Silhouette'
    plot_cluster_panel(ax, pca_xy, lbl, title, mstr)

plt.suptitle('Surface Environment Clusters — Best Silhouette Score\n'
             'Calmodulin apo vs holo, n=5000 subsampled CS points',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('clusters_best_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Best Davies-Bouldin ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, tag, pca_xy in [(axes[0], 'apo',  pca_apo),
                         (axes[1], 'holo', pca_holo)]:
    k      = best_configs[tag]['db']
    lbl    = labels[(tag, 'db')]
    r      = next(r for r in results[tag] if r['k'] == k)
    mstr   = f'k={k} | Silhouette={r["sil"]:.4f} | DB={r["db"]:.4f}'
    title  = f'{"Apo (1CFD)" if tag=="apo" else "Holo (1CLL)"} — Best Davies-Bouldin'
    plot_cluster_panel(ax, pca_xy, lbl, title, mstr)

plt.suptitle('Surface Environment Clusters — Best Davies-Bouldin Score\n'
             'Calmodulin apo vs holo, n=5000 subsampled CS points',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('clusters_best_db.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# ── Config: run for both best-sil and best-db ─────────────────────────────────
configs_to_plot = [
    ('sil', 'Best Silhouette', {'apo': 3,  'holo': 4}),
    ('db',  'Best Davies-Bouldin', {'apo': 6, 'holo': 11}),
]

# Rebuild subsampled data with same seed
rng      = np.random.default_rng(42)
n_sub    = 5000
idx_apo  = rng.choice(len(maps1.data), n_sub, replace=False)
idx_holo = rng.choice(len(maps2.data), n_sub, replace=False)
data_apo  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo = maps2.data.iloc[idx_holo].reset_index(drop=True)

pca_apo   = data_apo[['pca0',  'pca1']].values
pca_holo  = data_holo[['pca0', 'pca1']].values
xyz_apo   = data_apo[['x',  'y',  'z']].values
xyz_holo  = data_holo[['x', 'y', 'z']].values

def get_labels(results_list, k):
    return next(r['labels'] for r in results_list if r['k'] == k)


for metric_key, metric_title, best_k in configs_to_plot:

    lbl_apo  = get_labels(results['apo'],  best_k['apo'])
    lbl_holo = get_labels(results['holo'], best_k['holo'])

    k_apo    = best_k['apo']
    k_holo   = best_k['holo']
    k_max    = max(k_apo, k_holo)

    # Shared colormap so same cluster index = same color across panels
    cmap     = plt.cm.tab20
    colors   = [cmap(i / k_max) for i in range(k_max)]

    # ── Helper: build graph ────────────────────────────────────────────────────
    def build_graph(pca_xy, cluster_labels, k):
        centroids = np.array([pca_xy[cluster_labels == i].mean(axis=0)
                               for i in range(k)])
        # Weight = number of points shared between the two clusters
        # (simple proxy for boundary density)
        adj = np.zeros((k, k))
        for i in range(k):
            for j in range(i + 1, k):
                w = np.sum((cluster_labels == i) | (cluster_labels == j))
                adj[i, j] = adj[j, i] = w
        G   = nx.from_numpy_array(adj, create_using=nx.Graph)
        pos = {i: centroids[i] for i in range(k)}
        return G, pos, centroids, adj

    G_apo,  pos_apo,  cent_apo,  adj_apo  = build_graph(pca_apo,  lbl_apo,  k_apo)
    G_holo, pos_holo, cent_holo, adj_holo = build_graph(pca_holo, lbl_holo, k_holo)

    # ══════════════════════════════════════════════════════════════════════════
    # FIGURE 1 — Connectivity only (large, clear nodes)
    # ══════════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, G, pos, adj, k, tag, title_tag in [
        (axes[0], G_apo,  pos_apo,  adj_apo,  k_apo,  'apo',  'Apo (1CFD)'),
        (axes[1], G_holo, pos_holo, adj_holo, k_holo, 'holo', 'Holo (1CLL)'),
    ]:
        node_colors = [colors[i] for i in range(k)]

        # Edge widths scaled to weight
        edge_weights = [adj[u, v] / 1500 for u, v in G.edges()]

        nx.draw_networkx_edges(G, pos, width=edge_weights, ax=ax,
                               alpha=0.5, edge_color='#444444')
        nx.draw_networkx_nodes(G, pos, ax=ax,
                               node_size=800,
                               node_color=node_colors,
                               edgecolors='k', linewidths=1.5)
        nx.draw_networkx_labels(G, pos, ax=ax,
                                font_size=11, font_color='white',
                                font_weight='bold')

        # Annotate each node with cluster size
        n_total = len(lbl_apo if tag == 'apo' else lbl_holo)
        lbl_arr = lbl_apo if tag == 'apo' else lbl_holo
        for i in range(k):
            x_c, y_c = pos[i]
            pct = 100 * np.sum(lbl_arr == i) / n_total
            ax.text(x_c, y_c - 0.18, f'{pct:.1f}%',
                    ha='center', va='top', fontsize=7.5, color='#222222')

        ax.set_title(f'{title_tag} — k={k}', fontsize=12, fontweight='bold')
        ax.set_xlabel('PC1', fontsize=10)
        ax.set_ylabel('PC2', fontsize=10)
        ax.tick_params(left=True, bottom=True,
                       labelleft=True, labelbottom=True)
        ax.set_aspect('equal')
        ax.axis('on')

    plt.suptitle(f'Cluster Connectivity Graph — {metric_title}\n'
                 f'Node size proportional to display, % = fraction of CS points',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'connectivity_{metric_key}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ══════════════════════════════════════════════════════════════════════════
    # FIGURE 2 — 3D protein surface colored by cluster (MapSy style)
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(18, 8))

    for col, (xyz, lbl, k, tag, title_tag) in enumerate([
        (xyz_apo,  lbl_apo,  k_apo,  'apo',  'Apo (1CFD)'),
        (xyz_holo, lbl_holo, k_holo, 'holo', 'Holo (1CLL)'),
    ]):
        ax = fig.add_subplot(1, 2, col + 1, projection='3d')

        for i in range(k):
            mask = lbl == i
            ax.scatter(xyz[mask, 0], xyz[mask, 1], xyz[mask, 2],
                       color=colors[i], alpha=0.25, s=6,
                       label=f'Cluster {i} ({100*mask.sum()/len(lbl):.1f}%)',
                       rasterized=True)

        ax.set_xlabel('x (Å)', fontsize=9, labelpad=4)
        ax.set_ylabel('y (Å)', fontsize=9, labelpad=4)
        ax.set_zlabel('z (Å)', fontsize=9, labelpad=4)
        ax.set_title(f'{title_tag} — k={k}\nSurface environment clusters',
                     fontsize=11, fontweight='bold')

        # Compact legend outside
        ax.legend(loc='upper left', bbox_to_anchor=(0.0, 1.0),
                  fontsize=7, markerscale=2, framealpha=0.7,
                  ncol=2 if k > 6 else 1)

    plt.suptitle(f'MapSy-style 3D Surface Clustering — {metric_title}\n'
                 f'Calmodulin apo vs holo, n=5000 subsampled CS points',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'3d_clusters_{metric_key}.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Done — 4 figures saved:")
print("  connectivity_sil.png / connectivity_db.png")
print("  3d_clusters_sil.png  / 3d_clusters_db.png")

In [ ]:

# OR load from wherever you saved them — if not saved, recompute first:
# elec_apo, hydro_apo   = compute_electrostatic(atoms_apo),  compute_hydrophobicity(atoms_apo)
# elec_holo, hydro_holo = compute_electrostatic(atoms_holo), compute_hydrophobicity(atoms_holo)

# ── Subsample physicochemical with same indices ────────────────────────────────
# IMPORTANT: same rng seed and same idx_apo/idx_holo as clustering
rng      = np.random.default_rng(42)
n_sub    = 5000
idx_apo  = rng.choice(len(maps1.data), n_sub, replace=False)
idx_holo = rng.choice(len(maps2.data), n_sub, replace=False)

# Assuming full arrays exist — subset to same points as clustering
# Subsample physicochemical with same indices as clustering
elec_apo_sub   = elec_apo[idx_apo]
elec_holo_sub  = elec_holo[idx_holo]
hydro_apo_sub  = hydro_apo[idx_apo]
hydro_holo_sub = hydro_holo[idx_holo]

# Shared color scales
vmin_elec  = np.percentile(np.concatenate([elec_apo_sub,  elec_holo_sub]),   2)
vmax_elec  = np.percentile(np.concatenate([elec_apo_sub,  elec_holo_sub]),  98)
vmin_hydro = np.percentile(np.concatenate([hydro_apo_sub, hydro_holo_sub]),  2)
vmax_hydro = np.percentile(np.concatenate([hydro_apo_sub, hydro_holo_sub]), 98)

configs_to_plot = [
    ('sil', 'Best Silhouette',    {'apo': 3,  'holo': 4}),
    ('db',  'Best Davies-Bouldin', {'apo': 6, 'holo': 11}),
]

for metric_key, metric_title, best_k in configs_to_plot:

    lbl_apo  = get_labels(results['apo'],  best_k['apo'])
    lbl_holo = get_labels(results['holo'], best_k['holo'])
    k_max    = max(best_k['apo'], best_k['holo'])
    colors   = [plt.cm.tab20(i / k_max) for i in range(k_max)]

    # 3 rows x 2 cols
    fig, axes = plt.subplots(3, 2, figsize=(16, 22),
                             subplot_kw={'projection': '3d'})
    fig.suptitle(
        f'ACSF Clusters vs Physicochemical — {metric_title}\n'
        f'Clustering on 52 ACSF features (geometric only), '
        f'physicochemical computed independently',
        fontsize=13, fontweight='bold', y=0.99)

    row_configs = [
        (lbl_apo,       lbl_holo,       None,      None,       None,       'ACSF Clusters',  True),
        (elec_apo_sub,  elec_holo_sub,  'RdBu_r',  vmin_elec,  vmax_elec,  'Electrostatic',  False),
        (hydro_apo_sub, hydro_holo_sub, 'RdYlGn',  vmin_hydro, vmax_hydro, 'Hydrophobicity', False),
    ]

    for row, (val_apo, val_holo, cmap, vmin, vmax, row_label, discrete) in \
            enumerate(row_configs):

        for col, (xyz, vals, lbl, k, struct_label) in enumerate([
            (xyz_apo,  val_apo,  lbl_apo,  best_k['apo'],  'Apo (1CFD)'),
            (xyz_holo, val_holo, lbl_holo, best_k['holo'], 'Holo (1CLL)'),
        ]):
            ax = axes[row, col]

            if discrete:
                for i in range(k):
                    mask = vals == i
                    ax.scatter(xyz[mask, 0], xyz[mask, 1], xyz[mask, 2],
                               color=colors[i], alpha=0.25, s=6,
                               label=f'C{i} ({100*mask.sum()/len(vals):.0f}%)',
                               rasterized=True)
                if col == 1:
                    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0),
                              fontsize=7, markerscale=2,
                              ncol=2 if k > 5 else 1)
            else:
                sc = ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2],
                                c=vals, cmap=cmap, vmin=vmin, vmax=vmax,
                                alpha=0.25, s=6, rasterized=True)
                if col == 1:
                    plt.colorbar(sc, ax=ax, shrink=0.45, pad=0.15,
                                 label=row_label)

            ax.set_title(f'{struct_label} — {row_label}',
                         fontsize=10, fontweight='bold')
            ax.set_xlabel('x (Å)', fontsize=8, labelpad=2)
            ax.set_ylabel('y (Å)', fontsize=8, labelpad=2)
            ax.set_zlabel('z (Å)', fontsize=8, labelpad=2)
            ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.savefig(f'cluster_vs_physicochemical_{metric_key}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: cluster_vs_physicochemical_{metric_key}.png")

In [ ]:
from scipy import stats

def eta_squared(values, labels):
    """Fraction of variance in 'values' explained by cluster labels."""
    grand_mean = values.mean()
    k          = len(np.unique(labels))
    ss_between = sum(
        np.sum(labels == i) * (values[labels == i].mean() - grand_mean) ** 2
        for i in np.unique(labels)
    )
    ss_total   = np.sum((values - grand_mean) ** 2)
    return ss_between / ss_total

for metric_key, metric_title, best_k in configs_to_plot:
    lbl_apo  = get_labels(results['apo'],  best_k['apo'])
    lbl_holo = get_labels(results['holo'], best_k['holo'])

    print(f"\n{metric_title}")
    print(f"{'Descriptor':<20} {'Apo η²':>10} {'Holo η²':>10}  interpretation")
    print("-" * 65)
    for desc, val_a, val_h in [
        ('Electrostatic',  elec_apo_sub,  elec_holo_sub),
        ('Hydrophobicity', hydro_apo_sub, hydro_holo_sub),
    ]:
        eta_a = eta_squared(val_a, lbl_apo)
        eta_h = eta_squared(val_h, lbl_holo)
        interp = "strong overlap" if max(eta_a, eta_h) > 0.1 else \
                 "moderate"       if max(eta_a, eta_h) > 0.05 else "independent"
        print(f"{desc:<20} {eta_a:>10.4f} {eta_h:>10.4f}  {interp}")

In [ ]:
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import KBinsDiscretizer

def nmi_cluster_descriptor(values, labels, n_bins=10):
    """Normalized mutual information between cluster labels
    and binned physicochemical values. Range 0-1."""
    binner    = KBinsDiscretizer(n_bins=n_bins, encode='ordinal',
                                  strategy='quantile')
    vals_binned = binner.fit_transform(values.reshape(-1, 1)).ravel().astype(int)
    return normalized_mutual_info_score(labels, vals_binned)

for metric_key, metric_title, best_k in configs_to_plot:
    lbl_apo  = get_labels(results['apo'],  best_k['apo'])
    lbl_holo = get_labels(results['holo'], best_k['holo'])

    print(f"\n{metric_title}")
    print(f"{'Descriptor':<20} {'Apo NMI':>10} {'Holo NMI':>10}")
    print("-" * 45)
    for desc, val_a, val_h in [
        ('Electrostatic',  elec_apo_sub,  elec_holo_sub),
        ('Hydrophobicity', hydro_apo_sub, hydro_holo_sub),
    ]:
        nmi_a = nmi_cluster_descriptor(val_a, lbl_apo)
        nmi_h = nmi_cluster_descriptor(val_h, lbl_holo)
        print(f"{desc:<20} {nmi_a:>10.4f} {nmi_h:>10.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (desc, val_apo, val_holo) in enumerate([
    ('Electrostatic',  elec_apo_sub,  elec_holo_sub),
    ('Hydrophobicity', hydro_apo_sub, hydro_holo_sub),
]):
    for col, (metric_key, metric_title, best_k) in enumerate(configs_to_plot):
        ax = axes[row, col]

        lbl_apo  = get_labels(results['apo'],  best_k['apo'])
        lbl_holo = get_labels(results['holo'], best_k['holo'])
        k_max    = max(best_k['apo'], best_k['holo'])
        colors   = [plt.cm.tab20(i / k_max) for i in range(k_max)]

        # Apo clusters
        data_by_cluster = [val_apo[lbl_apo == i]
                           for i in range(best_k['apo'])]
        bp = ax.boxplot(data_by_cluster, patch_artist=True,
                        positions=np.arange(best_k['apo']) - 0.2,
                        widths=0.35, showfliers=False)
        for patch, c in zip(bp['boxes'], colors[:best_k['apo']]):
            patch.set_facecolor((*c[:3], 0.6))
            patch.set_edgecolor('k')

        # Holo clusters
        data_by_cluster_h = [val_holo[lbl_holo == i]
                              for i in range(best_k['holo'])]
        bp2 = ax.boxplot(data_by_cluster_h, patch_artist=True,
                         positions=np.arange(best_k['holo']) + 0.2,
                         widths=0.35, showfliers=False)
        for patch, c in zip(bp2['boxes'], colors[:best_k['holo']]):
            patch.set_facecolor((*c[:3], 0.3))
            patch.set_edgecolor('k')
            patch.set_linestyle('--')

        ax.set_xlabel('Cluster index', fontsize=10)
        ax.set_ylabel(desc, fontsize=10)
        ax.set_title(f'{desc} by cluster — {metric_title}', fontsize=10,
                     fontweight='bold')
        ax.axhline(np.concatenate([val_apo, val_holo]).mean(),
                   color='gray', linestyle=':', alpha=0.7,
                   label='Global mean')
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(facecolor='steelblue', label='Apo clusters'),
                            Patch(facecolor='lightblue', linestyle='--',
                                  label='Holo clusters')],
                  fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle('Physicochemical Values per Geometric Cluster\n'
             'Solid=Apo, Dashed=Holo — deviation from global mean indicates overlap',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('physicochemical_per_cluster_boxplot.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(7, 2, figsize=(16, 35))

descriptors = [
    ('Electrostatic',    elec_apo_sub,                        elec_holo_sub),
    ('Hydrophobicity',   hydro_apo_sub,                       hydro_holo_sub),
    ('RS_r12.0_004',     data_apo_sub['ACSF_RS_r12.0_004'].values, data_holo_sub['ACSF_RS_r12.0_004'].values),
    ('RS_r12.0_001',     data_apo_sub['ACSF_RS_r12.0_001'].values, data_holo_sub['ACSF_RS_r12.0_001'].values),
    ('RS_r12.0_005',     data_apo_sub['ACSF_RS_r12.0_005'].values, data_holo_sub['ACSF_RS_r12.0_005'].values),
    ('RC_r12.0_003',     data_apo_sub['ACSF_RC_r12.0_003'].values, data_holo_sub['ACSF_RC_r12.0_003'].values),
    ('RS_r12.0_003',     data_apo_sub['ACSF_RS_r12.0_003'].values, data_holo_sub['ACSF_RS_r12.0_003'].values),
]

for row, (desc, val_apo, val_holo) in enumerate(descriptors):
    for col, (metric_key, metric_title, best_k) in enumerate(configs_to_plot):
        ax = axes[row, col]

        lbl_apo  = get_labels(results['apo'],  best_k['apo'])
        lbl_holo = get_labels(results['holo'], best_k['holo'])
        k_max    = max(best_k['apo'], best_k['holo'])
        colors   = [plt.cm.tab20(i / k_max) for i in range(k_max)]

        # Apo
        bp = ax.boxplot(
            [val_apo[lbl_apo == i] for i in range(best_k['apo'])],
            patch_artist=True,
            positions=np.arange(best_k['apo']) - 0.2,
            widths=0.35, showfliers=False)
        for patch, c in zip(bp['boxes'], colors[:best_k['apo']]):
            patch.set_facecolor((*c[:3], 0.6))
            patch.set_edgecolor('k')

        # Holo
        bp2 = ax.boxplot(
            [val_holo[lbl_holo == i] for i in range(best_k['holo'])],
            patch_artist=True,
            positions=np.arange(best_k['holo']) + 0.2,
            widths=0.35, showfliers=False)
        for patch, c in zip(bp2['boxes'], colors[:best_k['holo']]):
            patch.set_facecolor((*c[:3], 0.3))
            patch.set_edgecolor('k')
            patch.set_linestyle('--')

        ax.axhline(np.concatenate([val_apo, val_holo]).mean(),
                   color='gray', linestyle=':', alpha=0.7)
        ax.set_xlabel('Cluster index', fontsize=9)
        ax.set_ylabel(desc, fontsize=9)
        ax.set_title(f'{desc} — {metric_title}', fontsize=9, fontweight='bold')
        ax.legend(handles=[Patch(facecolor='steelblue', alpha=0.6, label='Apo'),
                            Patch(facecolor='steelblue', alpha=0.3,
                                  linestyle='--', label='Holo')],
                  fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle('Descriptor Values per Geometric Cluster\n'
             'Solid=Apo, Dashed=Holo | Gray line=global mean',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('descriptor_per_cluster_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

def eta_squared(values, labels):
    grand_mean = values.mean()
    ss_between = sum(
        np.sum(labels == i) * (values[labels == i].mean() - grand_mean) ** 2
        for i in np.unique(labels)
    )
    ss_total = np.sum((values - grand_mean) ** 2)
    return ss_between / ss_total

from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import KBinsDiscretizer

def nmi_cluster_descriptor(values, labels, n_bins=10):
    binner      = KBinsDiscretizer(n_bins=n_bins, encode='ordinal',
                                   strategy='quantile')
    vals_binned = binner.fit_transform(
                      values.reshape(-1, 1)).ravel().astype(int)
    return normalized_mutual_info_score(labels, vals_binned)

descriptors = [
    ('Electrostatic',  elec_apo_sub,                              elec_holo_sub),
    ('Hydrophobicity', hydro_apo_sub,                             hydro_holo_sub),
    ('RS_r12.0_004',   data_apo_sub['ACSF_RS_r12.0_004'].values,  data_holo_sub['ACSF_RS_r12.0_004'].values),
    ('RS_r12.0_001',   data_apo_sub['ACSF_RS_r12.0_001'].values,  data_holo_sub['ACSF_RS_r12.0_001'].values),
    ('RS_r12.0_005',   data_apo_sub['ACSF_RS_r12.0_005'].values,  data_holo_sub['ACSF_RS_r12.0_005'].values),
    ('RC_r12.0_003',   data_apo_sub['ACSF_RC_r12.0_003'].values,  data_holo_sub['ACSF_RC_r12.0_003'].values),
    ('RS_r12.0_003',   data_apo_sub['ACSF_RS_r12.0_003'].values,  data_holo_sub['ACSF_RS_r12.0_003'].values),
]

# ── η² ────────────────────────────────────────────────────────────────────────
for metric_key, metric_title, best_k in configs_to_plot:
    lbl_apo  = get_labels(results['apo'],  best_k['apo'])
    lbl_holo = get_labels(results['holo'], best_k['holo'])

    print(f"\n{metric_title}")
    print(f"{'Descriptor':<20} {'Apo η²':>10} {'Holo η²':>10}  interpretation")
    print("-" * 65)
    for desc, val_a, val_h in descriptors:
        eta_a  = eta_squared(val_a, lbl_apo)
        eta_h  = eta_squared(val_h, lbl_holo)
        interp = "strong overlap" if max(eta_a, eta_h) > 0.1 else \
                 "moderate"       if max(eta_a, eta_h) > 0.05 else "independent"
        print(f"{desc:<20} {eta_a:>10.4f} {eta_h:>10.4f}  {interp}")

# ── NMI ───────────────────────────────────────────────────────────────────────
for metric_key, metric_title, best_k in configs_to_plot:
    lbl_apo  = get_labels(results['apo'],  best_k['apo'])
    lbl_holo = get_labels(results['holo'], best_k['holo'])

    print(f"\n{metric_title}")
    print(f"{'Descriptor':<20} {'Apo NMI':>10} {'Holo NMI':>10}")
    print("-" * 45)
    for desc, val_a, val_h in descriptors:
        nmi_a = nmi_cluster_descriptor(val_a, lbl_apo)
        nmi_h = nmi_cluster_descriptor(val_h, lbl_holo)
        print(f"{desc:<20} {nmi_a:>10.4f} {nmi_h:>10.4f}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def _atom_color(elem):
    return {'C': '#404040', 'N': '#4169E1', 'O': '#FF4444',
            'S': '#FFD700', 'CA': '#808080'}.get(elem, '#AAAAAA')

def plot_protein_cs(atoms, maps, feat, title):
    fig = go.Figure()

    # Protein atoms
    pos = atoms.get_positions()
    sym = np.array(atoms.get_chemical_symbols())
    for elem in sorted(set(sym)):
        mask = sym == elem
        p    = pos[mask]
        fig.add_trace(go.Scatter3d(
            x=p[:, 0], y=p[:, 1], z=p[:, 2],
            mode='markers', name=elem,
            marker=dict(size=3, color=_atom_color(elem),
                        opacity=0.4, line=dict(width=0.3, color='black'))))

    # CS points colored by feature
    cvals = maps.data[feat].values
    fig.add_trace(go.Scatter3d(
        x=maps.data['x'], y=maps.data['y'], z=maps.data['z'],
        mode='markers', name='Contact Space',
        marker=dict(size=4, color=cvals, colorscale='RdBu_r',
                    opacity=0.75,
                    colorbar=dict(title=feat, thickness=14),
                    showscale=True),
        showlegend=False))

    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data',
                   xaxis_title='x (Å)', yaxis_title='y (Å)',
                   zaxis_title='z (Å)'),
        width=950, height=720, template='plotly_white')
    return fig


def plot_realspace_pca(atoms, maps, feat, title):
    df    = maps.data
    cvals = df[feat].values
    cmin, cmax = np.percentile(cvals, 2), np.percentile(cvals, 98)

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=[f'Real space — {feat}', 'PCA space'],
        horizontal_spacing=0.02)

    # Protein atoms (real space only)
    pos = atoms.get_positions()
    sym = np.array(atoms.get_chemical_symbols())
    for elem in sorted(set(sym)):
        mask = sym == elem
        p    = pos[mask]
        fig.add_trace(go.Scatter3d(
            x=p[:, 0], y=p[:, 1], z=p[:, 2],
            mode='markers', name=elem,
            marker=dict(size=3, color=_atom_color(elem), opacity=0.3)),
            row=1, col=1)

    # CS points — real space
    fig.add_trace(go.Scatter3d(
        x=df['x'], y=df['y'], z=df['z'],
        mode='markers', name='CS real',
        marker=dict(size=4, color=cvals, colorscale='RdBu_r',
                    cmin=cmin, cmax=cmax, opacity=0.7, showscale=False),
        showlegend=False), row=1, col=1)

    # CS points — PCA space
    fig.add_trace(go.Scatter3d(
        x=df['pca0'], y=df['pca1'], z=df['pca2'],
        mode='markers', name='CS pca',
        marker=dict(size=4, color=cvals, colorscale='RdBu_r',
                    cmin=cmin, cmax=cmax, opacity=0.7,
                    colorbar=dict(title=feat, thickness=14, x=1.01),
                    showscale=True),
        showlegend=False), row=1, col=2)

    fig.update_layout(
        title_text=title, width=1350, height=660,
        template='plotly_white',
        scene=dict(xaxis_title='x (Å)', yaxis_title='y (Å)',
                   zaxis_title='z (Å)', aspectmode='data'),
        scene2=dict(xaxis_title='PC1', yaxis_title='PC2',
                    zaxis_title='PC3', aspectmode='cube'))
    return fig


# ── Run for features of interest ─────────────────────────────────────────────
maps1.data['electrostatic']  = elec_apo
maps1.data['hydrophobicity'] = hydro_apo
maps2.data['electrostatic']  = elec_holo
maps2.data['hydrophobicity'] = hydro_holo
features_to_plot = [
    'electrostatic',
    'hydrophobicity',
    'ACSF_RS_r12.0_001',
    'ACSF_RS_r12.0_004',
]

for feat in features_to_plot:
    plot_protein_cs(
        calmodulin_apo, maps1, feat,
        f'Apo (1CFD) — {feat}').show()
    plot_protein_cs(
        calmodulin_holo, maps2, feat,
        f'Holo (1CLL) — {feat}').show()
    plot_realspace_pca(
        calmodulin_apo, maps1, feat,
        f'Apo (1CFD) — Real space vs PCA — {feat}').show()
    plot_realspace_pca(
        calmodulin_holo, maps2, feat,
        f'Holo (1CLL) — Real space vs PCA — {feat}').show()

In [ ]:
# ── Verify alignment first ────────────────────────────────────────────────────
rng      = np.random.default_rng(42)  # same seed as k-sweep
idx_apo  = rng.choice(len(maps1.data), 5000, replace=False)
idx_holo = rng.choice(len(maps2.data), 5000, replace=False)

# Keep reset_index so positional indexing is safe
data_apo_sub  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo_sub = maps2.data.iloc[idx_holo].reset_index(drop=True)

lbl_apo_sil  = get_labels(results['apo'],  3)
lbl_holo_sil = get_labels(results['holo'], 4)
lbl_apo_db   = get_labels(results['apo'],  6)
lbl_holo_db  = get_labels(results['holo'], 11)

# Explicit alignment checks
assert len(lbl_apo_sil)  == len(data_apo_sub),  \
    f"APO sil label mismatch: {len(lbl_apo_sil)} vs {len(data_apo_sub)}"
assert len(lbl_holo_sil) == len(data_holo_sub), \
    f"HOLO sil label mismatch: {len(lbl_holo_sil)} vs {len(data_holo_sub)}"
assert len(lbl_apo_db)   == len(data_apo_sub),  \
    f"APO db label mismatch: {len(lbl_apo_db)} vs {len(data_apo_sub)}"
assert len(lbl_holo_db)  == len(data_holo_sub), \
    f"HOLO db label mismatch: {len(lbl_holo_db)} vs {len(data_holo_sub)}"
print("All alignment checks passed.")

# ── Color scales ──────────────────────────────────────────────────────────────
elec_all  = np.concatenate([maps1.data['electrostatic'].values,
                             maps2.data['electrostatic'].values])
hydro_all = np.concatenate([maps1.data['hydrophobicity'].values,
                             maps2.data['hydrophobicity'].values])
vmin_e, vmax_e = np.percentile(elec_all,  [2, 98])
vmin_h, vmax_h = np.percentile(hydro_all, [2, 98])

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 28))
fig.suptitle(
    'MapSy — Calmodulin Surface Characterization\n'
    'Apo (1CFD) vs Holo (1CLL)\n'
    'Row 1: Electrostatic (all CS points) · '
    'Row 2: Hydrophobicity (all CS points)\n'
    'Row 3: ACSF Clusters — Best Silhouette (n=5000) · '
    'Row 4: ACSF Clusters — Best Davies-Bouldin (n=5000)',
    fontsize=12, fontweight='bold', y=0.99)

def add_protein_atoms(ax, atoms, alpha=0.08):
    pos = atoms.get_positions()
    sym = np.array(atoms.get_chemical_symbols())
    color_map = {'C': '#404040', 'N': '#4169E1',
                 'O': '#FF4444', 'S': '#FFD700'}
    for elem in sorted(set(sym)):
        mask = sym == elem
        p    = pos[mask]
        ax.scatter(p[:, 0], p[:, 1], p[:, 2],
                   c=color_map.get(elem, '#AAAAAA'),
                   s=1, alpha=alpha, rasterized=True)

plot_configs = [
    (1, 'electrostatic',  'RdBu_r',  vmin_e, vmax_e,
     'Electrostatic (r_cut=8Å, AMBER ff14SB)',
     False, None, None, None, None),

    (2, 'hydrophobicity', 'RdYlGn',  vmin_h, vmax_h,
     'Hydrophobicity (r_cut=5Å, Kyte-Doolittle)',
     False, None, None, None, None),

    (3, None, None, None, None,
     'ACSF Clusters — Best Silhouette\n'
     'Apo k=3 (Sil=0.142) | Holo k=4 (Sil=0.139)\n'
     'n=5000 subsampled, labels aligned to same subsample',
     True, lbl_apo_sil, lbl_holo_sil, 3, 4),

    (4, None, None, None, None,
     'ACSF Clusters — Best Davies-Bouldin\n'
     'Apo k=6 (DB=1.782) | Holo k=11 (DB=1.841)\n'
     'n=5000 subsampled, labels aligned to same subsample',
     True, lbl_apo_db, lbl_holo_db, 6, 11),
]

for row, feat, cmap, vmin, vmax, row_label, discrete, \
        lbl_a, lbl_h, k_a, k_h in plot_configs:

    for col, (atoms, maps_obj, data_sub, lbl, k, struct_label) in enumerate([
        (calmodulin_apo,  maps1, data_apo_sub,  lbl_a, k_a, 'Apo (1CFD)'),
        (calmodulin_holo, maps2, data_holo_sub, lbl_h, k_h, 'Holo (1CLL)'),
    ]):
        ax = fig.add_subplot(4, 2, (row - 1) * 2 + col + 1,
                             projection='3d')
        add_protein_atoms(ax, atoms)

        if not discrete:
            # All CS points for continuous descriptors
            vals = maps_obj.data[feat].values
            sc   = ax.scatter(
                maps_obj.data['x'].values,
                maps_obj.data['y'].values,
                maps_obj.data['z'].values,
                c=vals, cmap=cmap, vmin=vmin, vmax=vmax,
                s=3, alpha=0.7, rasterized=True)
            if col == 1:
                plt.colorbar(sc, ax=ax, shrink=0.5, pad=0.1,
                             label=feat)
        else:
            # Subsampled points for clusters — labels positionally
            # aligned to data_sub (both reset_index to 0..4999)
            xyz    = data_sub[['x', 'y', 'z']].values  # shape (5000, 3)
            colors = plt.cm.tab20(np.linspace(0, 1, k))
            for i in range(k):
                mask = lbl == i                          # positional mask
                pct  = 100 * mask.sum() / len(lbl)
                ax.scatter(xyz[mask, 0], xyz[mask, 1], xyz[mask, 2],
                           color=colors[i], s=4, alpha=0.75,
                           label=f'C{i} ({pct:.0f}%)',
                           rasterized=True)
            if col == 1:
                ax.legend(loc='upper left',
                          bbox_to_anchor=(1.15, 1.0),
                          fontsize=6, markerscale=2,
                          ncol=2 if k > 6 else 1,
                          title=f'Clusters k={k}',
                          title_fontsize=7)

        ax.set_title(f'{struct_label}\n{row_label}',
                     fontsize=7, fontweight='bold', pad=2)
        ax.set_xlabel('x (Å)', fontsize=7, labelpad=1)
        ax.set_ylabel('y (Å)', fontsize=7, labelpad=1)
        ax.set_zlabel('z (Å)', fontsize=7, labelpad=1)
        ax.tick_params(labelsize=6)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('calmodulin_teaser.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: calmodulin_teaser.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from umap import UMAP

# ── UMAP on subsampled data ───────────────────────────────────────────────────
# Use same subsample as clustering
rng      = np.random.default_rng(42)
idx_apo  = rng.choice(len(maps1.data), 5000, replace=False)
idx_holo = rng.choice(len(maps2.data), 5000, replace=False)

data_apo_sub  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo_sub = maps2.data.iloc[idx_holo].reset_index(drop=True)

acsf_features = maps1.features
scaler        = StandardScaler()
X_apo_sc      = scaler.fit_transform(data_apo_sub[acsf_features].values)
X_holo_sc     = scaler.transform(data_holo_sub[acsf_features].values)

# Fit UMAP on apo, transform both
print("Fitting UMAP...")
reducer    = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                  random_state=42, n_jobs=4)
umap_apo   = reducer.fit_transform(X_apo_sc)
umap_holo  = reducer.transform(X_holo_sc)
print("UMAP done.")

# ── DBCV via HDBSCAN ─────────────────────────────────────────────────────────
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("hdbscan not installed — skipping DBCV. Run: pip install hdbscan")

if HAS_HDBSCAN:
    print("Fitting HDBSCAN + DBCV...")
    # Run HDBSCAN on UMAP embedding (faster, captures manifold structure)
    hdb_apo  = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=10,
                                 gen_min_span_tree=True)
    hdb_holo = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=10,
                                 gen_min_span_tree=True)
    hdb_apo.fit(umap_apo)
    hdb_holo.fit(umap_holo)

    lbl_hdb_apo  = hdb_apo.labels_
    lbl_hdb_holo = hdb_holo.labels_

    dbcv_apo  = hdbscan.validity.validity_index(
                    umap_apo.astype(np.float64), lbl_hdb_apo)
    dbcv_holo = hdbscan.validity.validity_index(
                    umap_holo.astype(np.float64), lbl_hdb_holo)

    n_clusters_apo  = len(set(lbl_hdb_apo))  - (1 if -1 in lbl_hdb_apo  else 0)
    n_clusters_holo = len(set(lbl_hdb_holo)) - (1 if -1 in lbl_hdb_holo else 0)
    noise_apo  = 100 * (lbl_hdb_apo  == -1).sum() / len(lbl_hdb_apo)
    noise_holo = 100 * (lbl_hdb_holo == -1).sum() / len(lbl_hdb_holo)

    print(f"\nHDBSCAN + DBCV results:")
    print(f"  Apo:  {n_clusters_apo} clusters, "
          f"noise={noise_apo:.1f}%, DBCV={dbcv_apo:.4f}")
    print(f"  Holo: {n_clusters_holo} clusters, "
          f"noise={noise_holo:.1f}%, DBCV={dbcv_holo:.4f}")
    print(f"  DBCV range: -1 (worst) to 1 (best)")

# ── Pull spectral labels for comparison ───────────────────────────────────────
lbl_apo_sil  = get_labels(results['apo'],  3)
lbl_holo_sil = get_labels(results['holo'], 4)
lbl_apo_db   = get_labels(results['apo'],  6)
lbl_holo_db  = get_labels(results['holo'], 11)

# ── Plot: UMAP colored by spectral clusters + HDBSCAN ────────────────────────
n_rows = 4 if HAS_HDBSCAN else 3
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 5 * n_rows))
fig.suptitle('UMAP Embedding — Calmodulin Surface Environments\n'
             '52 ACSF features, n=5000 subsampled CS points per structure',
             fontsize=13, fontweight='bold')

row_configs = [
    ('Apo/Holo label\n(distribution shift)',
     np.zeros(5000), np.ones(5000),
     'RdBu_r', False, 'Class (0=Apo, 1=Holo)'),
    ('Spectral Clusters — Best Silhouette\n'
     'Apo k=3, Holo k=4',
     lbl_apo_sil, lbl_holo_sil, 'tab20', True, 'Cluster'),
    ('Spectral Clusters — Best Davies-Bouldin\n'
     'Apo k=6, Holo k=11',
     lbl_apo_db, lbl_holo_db, 'tab20', True, 'Cluster'),
]
if HAS_HDBSCAN:
    row_configs.append((
        'HDBSCAN Clusters (density-based, no shape assumption)\n'
        'Labels -1 = noise points',
        lbl_hdb_apo, lbl_hdb_holo, 'tab20', True, 'Cluster'))

for row, (title, lbl_a, lbl_h, cmap, discrete, cbar_label) in \
        enumerate(row_configs):
    for col, (umap_xy, lbl, struct) in enumerate([
        (umap_apo,  lbl_a, 'Apo (1CFD)'),
        (umap_holo, lbl_h, 'Holo (1CLL)'),
    ]):
        ax = axes[row, col]

        if discrete:
            k      = len(np.unique(lbl[lbl >= 0]))
            colors = plt.cm.tab20(np.linspace(0, 1, max(k, 1)))
            # Noise points first in gray
            noise  = lbl == -1
            if noise.any():
                ax.scatter(umap_xy[noise, 0], umap_xy[noise, 1],
                           c='lightgray', s=4, alpha=0.3,
                           label='Noise', rasterized=True)
            for i in range(k):
                mask = lbl == i
                pct  = 100 * mask.sum() / len(lbl)
                ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
                           color=colors[i], s=4, alpha=0.5,
                           label=f'C{i} ({pct:.0f}%)',
                           rasterized=True)
            if col == 1 and k <= 12:
                ax.legend(fontsize=6, markerscale=2,
                          loc='upper right', ncol=2)
        else:
            sc = ax.scatter(umap_xy[:, 0], umap_xy[:, 1],
                            c=lbl, cmap=cmap, s=4, alpha=0.5,
                            rasterized=True)
            if col == 1:
                plt.colorbar(sc, ax=ax, label=cbar_label, shrink=0.7)

        ax.set_title(f'{struct}\n{title}', fontsize=8, fontweight='bold')
        ax.set_xlabel('UMAP 1', fontsize=8)
        ax.set_ylabel('UMAP 2', fontsize=8)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('umap_cluster_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("CLUSTERING VALIDATION SUMMARY")
print("="*60)
print(f"{'Metric':<35} {'Apo':>10} {'Holo':>10}  Note")
print("-"*60)
print(f"{'Spectral Sil (k=3/4)':<35} "
      f"{get_labels(results['apo'],  3) is not None and next(r['sil'] for r in results['apo']  if r['k']==3):>10.4f} "
      f"{next(r['sil'] for r in results['holo'] if r['k']==4):>10.4f}  spherical bias")
print(f"{'Spectral DB (k=6/11)':<35} "
      f"{next(r['db'] for r in results['apo']  if r['k']==6):>10.4f} "
      f"{next(r['db'] for r in results['holo'] if r['k']==11):>10.4f}  spherical bias")
if HAS_HDBSCAN:
    print(f"{'HDBSCAN DBCV':<35} {dbcv_apo:>10.4f} {dbcv_holo:>10.4f}"
          f"  density-based (-1 to 1)")
    print(f"{'HDBSCAN n_clusters':<35} {n_clusters_apo:>10} {n_clusters_holo:>10}")
    print(f"{'HDBSCAN noise %':<35} {noise_apo:>9.1f}% {noise_holo:>9.1f}%")
print(f"{'RF AUC (apo vs holo)':<35} {'0.805':>10} {'—':>10}  task-based ✓")
print(f"{'LDA Cohen d':<35} {'0.533':>10} {'—':>10}  task-based ✓")
print("="*60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from umap import UMAP
import hdbscan

# ── Setup — same seed, same subsample ────────────────────────────────────────
rng      = np.random.default_rng(42)
idx_apo  = rng.choice(len(maps1.data), 5000, replace=False)
idx_holo = rng.choice(len(maps2.data), 5000, replace=False)

data_apo_sub  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo_sub = maps2.data.iloc[idx_holo].reset_index(drop=True)

# ── Build feature matrices with physicochemical added ─────────────────────────
acsf_features = maps1.features
all_features  = list(acsf_features) + ['electrostatic', 'hydrophobicity']

# Add physicochemical to subsampled dataframes
data_apo_sub['electrostatic']  = elec_apo[idx_apo]
data_apo_sub['hydrophobicity'] = hydro_apo[idx_apo]
data_holo_sub['electrostatic'] = elec_holo[idx_holo]
data_holo_sub['hydrophobicity']= hydro_holo[idx_holo]

scaler    = StandardScaler()
X_apo_sc  = scaler.fit_transform(data_apo_sub[all_features].values)
X_holo_sc = scaler.transform(data_holo_sub[all_features].values)

print(f"Feature matrix: {X_apo_sc.shape[1]} features "
      f"(52 ACSF + 2 physicochemical)")

# ── UMAP ──────────────────────────────────────────────────────────────────────
print("Fitting UMAP (52 ACSF + physicochemical)...")
reducer       = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                     random_state=42, n_jobs=4)
umap_apo_full  = reducer.fit_transform(X_apo_sc)
umap_holo_full = reducer.transform(X_holo_sc)
print("UMAP done.")

# ── HDBSCAN + DBCV ────────────────────────────────────────────────────────────
print("Fitting HDBSCAN...")
hdb_apo_full  = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=10,
                                  gen_min_span_tree=True)
hdb_holo_full = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=10,
                                  gen_min_span_tree=True)
hdb_apo_full.fit(umap_apo_full)
hdb_holo_full.fit(umap_holo_full)

lbl_hdb_apo_full  = hdb_apo_full.labels_
lbl_hdb_holo_full = hdb_holo_full.labels_

dbcv_apo_full  = hdbscan.validity.validity_index(
                     umap_apo_full.astype(np.float64),  lbl_hdb_apo_full)
dbcv_holo_full = hdbscan.validity.validity_index(
                     umap_holo_full.astype(np.float64), lbl_hdb_holo_full)

n_cl_apo_full  = len(set(lbl_hdb_apo_full))  - \
                 (1 if -1 in lbl_hdb_apo_full  else 0)
n_cl_holo_full = len(set(lbl_hdb_holo_full)) - \
                 (1 if -1 in lbl_hdb_holo_full else 0)
noise_apo_full  = 100 * (lbl_hdb_apo_full  == -1).sum() / len(lbl_hdb_apo_full)
noise_holo_full = 100 * (lbl_hdb_holo_full == -1).sum() / len(lbl_hdb_holo_full)

print(f"\nHDBSCAN (54 features):")
print(f"  Apo:  {n_cl_apo_full}  clusters, "
      f"noise={noise_apo_full:.1f}%,  DBCV={dbcv_apo_full:.4f}")
print(f"  Holo: {n_cl_holo_full} clusters, "
      f"noise={noise_holo_full:.1f}%, DBCV={dbcv_holo_full:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
# Pull ACSF-only spectral labels for comparison in same figure
lbl_apo_sil  = get_labels(results['apo'],  3)
lbl_holo_sil = get_labels(results['holo'], 4)
lbl_apo_db   = get_labels(results['apo'],  6)
lbl_holo_db  = get_labels(results['holo'], 11)

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
fig.suptitle(
    'UMAP Embedding — Calmodulin Surface Environments\n'
    '54 features: 52 ACSF + Electrostatic + Hydrophobicity\n'
    '(compare to ACSF-only UMAP for effect of physicochemical descriptors)',
    fontsize=12, fontweight='bold')

row_configs = [
    ('Apo/Holo label\n(distribution shift — 54 features)',
     np.zeros(5000), np.ones(5000),
     'RdBu_r', False, 'Class (0=Apo, 1=Holo)'),

    ('Electrostatic value\n(continuous, shared colorscale)',
     elec_apo[idx_apo], elec_holo[idx_holo],
     'RdBu_r', False, 'Electrostatic'),

    ('Spectral Clusters (ACSF-only) — Best Silhouette\n'
     'Apo k=3, Holo k=4 — projected onto 54-feature UMAP',
     lbl_apo_sil, lbl_holo_sil, 'tab20', True, 'Cluster'),

    ('HDBSCAN (54 features, density-based)\n'
     f'Apo: {n_cl_apo_full} clusters, noise={noise_apo_full:.0f}%  |  '
     f'Holo: {n_cl_holo_full} clusters, noise={noise_holo_full:.0f}%',
     lbl_hdb_apo_full, lbl_hdb_holo_full, 'tab20', True, 'Cluster'),
]

elec_all  = np.concatenate([elec_apo[idx_apo], elec_holo[idx_holo]])
vmin_e, vmax_e = np.percentile(elec_all, [2, 98])

for row, (title, lbl_a, lbl_h, cmap, discrete, cbar_label) in \
        enumerate(row_configs):
    for col, (umap_xy, lbl, struct) in enumerate([
        (umap_apo_full,  lbl_a, 'Apo (1CFD)'),
        (umap_holo_full, lbl_h, 'Holo (1CLL)'),
    ]):
        ax = axes[row, col]

        if discrete:
            k      = len(np.unique(lbl[lbl >= 0]))
            colors = plt.cm.tab20(np.linspace(0, 1, max(k, 1)))
            noise  = lbl == -1
            if noise.any():
                ax.scatter(umap_xy[noise, 0], umap_xy[noise, 1],
                           c='lightgray', s=4, alpha=0.3,
                           label='Noise', rasterized=True)
            for i in range(k):
                mask = lbl == i
                pct  = 100 * mask.sum() / len(lbl)
                ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
                           color=colors[i], s=4, alpha=0.5,
                           label=f'C{i} ({pct:.0f}%)',
                           rasterized=True)
            if col == 1 and k <= 14:
                ax.legend(fontsize=6, markerscale=2,
                          loc='upper right', ncol=2)
        else:
            # For electrostatic use shared vmin/vmax
            kw = dict(vmin=vmin_e, vmax=vmax_e) \
                 if cbar_label == 'Electrostatic' else {}
            sc = ax.scatter(umap_xy[:, 0], umap_xy[:, 1],
                            c=lbl, cmap=cmap, s=4, alpha=0.5,
                            rasterized=True, **kw)
            if col == 1:
                plt.colorbar(sc, ax=ax, label=cbar_label, shrink=0.7)

        ax.set_title(f'{struct}\n{title}', fontsize=8, fontweight='bold')
        ax.set_xlabel('UMAP 1', fontsize=8)
        ax.set_ylabel('UMAP 2', fontsize=8)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('umap_full_features.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Comparison summary ────────────────────────────────────────────────────────
print("\n" + "="*65)
print("COMPARISON: ACSF-only vs ACSF + Physicochemical")
print("="*65)
print(f"{'Metric':<40} {'Apo':>10} {'Holo':>10}")
print("-"*65)
print(f"{'HDBSCAN n_clusters (ACSF only)':<40} {'13':>10} {'7':>10}")
print(f"{'HDBSCAN n_clusters (ACSF + physchem)':<40} "
      f"{n_cl_apo_full:>10} {n_cl_holo_full:>10}")
print(f"{'HDBSCAN noise % (ACSF only)':<40} {'38.3%':>10} {'56.8%':>10}")
print(f"{'HDBSCAN noise % (ACSF + physchem)':<40} "
      f"{noise_apo_full:>9.1f}% {noise_holo_full:>9.1f}%")
print(f"{'DBCV (ACSF only)':<40} {'-0.123':>10} {'-0.130':>10}")
print(f"{'DBCV (ACSF + physchem)':<40} "
      f"{dbcv_apo_full:>10.4f} {dbcv_holo_full:>10.4f}")
print("="*65)
print("\nKey question: does adding physicochemical descriptors")
print("reduce noise % or improve DBCV?")
print("If yes → physicochemical adds density structure to the manifold.")
print("If no  → geometry and chemistry encode independent information.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from umap import UMAP
import hdbscan

# ── Setup — same seed, same subsample ────────────────────────────────────────
rng      = np.random.default_rng(42)
idx_apo  = rng.choice(len(maps1.data), 5000, replace=False)
idx_holo = rng.choice(len(maps2.data), 5000, replace=False)

data_apo_sub  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo_sub = maps2.data.iloc[idx_holo].reset_index(drop=True)

# ── Build feature matrices with physicochemical added ─────────────────────────
acsf_features = maps1.features
all_features  = list(acsf_features) + ['electrostatic', 'hydrophobicity']

# Add physicochemical to subsampled dataframes
data_apo_sub['electrostatic']  = elec_apo[idx_apo]
data_apo_sub['hydrophobicity'] = hydro_apo[idx_apo]
data_holo_sub['electrostatic'] = elec_holo[idx_holo]
data_holo_sub['hydrophobicity']= hydro_holo[idx_holo]

# FIX 1: Scale Apo and Holo independently so Holo isn't distorted by Apo's distribution
scaler_apo  = StandardScaler()
scaler_holo = StandardScaler()

X_apo_sc  = scaler_apo.fit_transform(data_apo_sub[all_features].values)
X_holo_sc = scaler_holo.fit_transform(data_holo_sub[all_features].values)

print(f"Feature matrix: {X_apo_sc.shape[1]} features "
      f"(52 ACSF + 2 physicochemical)")

# ── UMAP ──────────────────────────────────────────────────────────────────────
print("Fitting UMAP (52 ACSF + physicochemical) independently...")

# FIX 1 (cont.): Fit UMAP independently to reveal each state's true natural manifold
reducer_apo  = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42, n_jobs=4)
reducer_holo = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42, n_jobs=4)

umap_apo_full  = reducer_apo.fit_transform(X_apo_sc)
umap_holo_full = reducer_holo.fit_transform(X_holo_sc)
print("UMAP done.")

# ── HDBSCAN + DBCV ────────────────────────────────────────────────────────────
print("Fitting HDBSCAN...")
hdb_apo_full  = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=10,
                                  gen_min_span_tree=True)
hdb_holo_full = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=10,
                                  gen_min_span_tree=True)

# We still fit HDBSCAN on the UMAP embeddings for visual clustering
hdb_apo_full.fit(umap_apo_full)
hdb_holo_full.fit(umap_holo_full)

lbl_hdb_apo_full  = hdb_apo_full.labels_
lbl_hdb_holo_full = hdb_holo_full.labels_

# FIX 2: Calculate DBCV strictly on the high-dimensional scaled data (X_..._sc), 
# NOT the 2D UMAP projection, to get a mathematically valid density score.
dbcv_apo_full  = hdbscan.validity.validity_index(
                     X_apo_sc.astype(np.float64), lbl_hdb_apo_full)
dbcv_holo_full = hdbscan.validity.validity_index(
                     X_holo_sc.astype(np.float64), lbl_hdb_holo_full)

n_cl_apo_full  = len(set(lbl_hdb_apo_full))  - \
                 (1 if -1 in lbl_hdb_apo_full  else 0)
n_cl_holo_full = len(set(lbl_hdb_holo_full)) - \
                 (1 if -1 in lbl_hdb_holo_full else 0)
noise_apo_full  = 100 * (lbl_hdb_apo_full  == -1).sum() / len(lbl_hdb_apo_full)
noise_holo_full = 100 * (lbl_hdb_holo_full == -1).sum() / len(lbl_hdb_holo_full)

print(f"\nHDBSCAN (54 features):")
print(f"  Apo:  {n_cl_apo_full}  clusters, "
      f"noise={noise_apo_full:.1f}%,  DBCV={dbcv_apo_full:.4f}")
print(f"  Holo: {n_cl_holo_full} clusters, "
      f"noise={noise_holo_full:.1f}%, DBCV={dbcv_holo_full:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
# Pull ACSF-only spectral labels for comparison in same figure
lbl_apo_sil  = get_labels(results['apo'],  3)
lbl_holo_sil = get_labels(results['holo'], 4)
lbl_apo_db   = get_labels(results['apo'],  6) 
lbl_holo_db  = get_labels(results['holo'], 11)

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
fig.suptitle(
    'UMAP Embedding — Calmodulin Surface Environments\n'
    '54 features: 52 ACSF + Electrostatic + Hydrophobicity\n'
    '(Independently scaled and projected to preserve true manifold topology)',
    fontsize=12, fontweight='bold')

row_configs = [
    ('Apo/Holo label\n(Shown for reference)',
     np.zeros(5000), np.ones(5000),
     'RdBu_r', False, 'Class (0=Apo, 1=Holo)'),

    ('Electrostatic value\n(continuous, shared colorscale)',
     elec_apo[idx_apo], elec_holo[idx_holo],
     'RdBu_r', False, 'Electrostatic'),

    ('Spectral Clusters (ACSF-only) — Best Silhouette\n'
     'Apo k=3, Holo k=4 — projected onto 54-feature UMAP',
     lbl_apo_sil, lbl_holo_sil, 'tab20', True, 'Cluster'),

    ('HDBSCAN (54 features, density-based)\n'
     f'Apo: {n_cl_apo_full} clusters, noise={noise_apo_full:.0f}%  |  '
     f'Holo: {n_cl_holo_full} clusters, noise={noise_holo_full:.0f}%',
     lbl_hdb_apo_full, lbl_hdb_holo_full, 'tab20', True, 'Cluster'),
]

elec_all  = np.concatenate([elec_apo[idx_apo], elec_holo[idx_holo]])
vmin_e, vmax_e = np.percentile(elec_all, [2, 98])

for row, (title, lbl_a, lbl_h, cmap, discrete, cbar_label) in \
        enumerate(row_configs):
    for col, (umap_xy, lbl, struct) in enumerate([
        (umap_apo_full,  lbl_a, 'Apo (1CFD)'),
        (umap_holo_full, lbl_h, 'Holo (1CLL)'),
    ]):
        ax = axes[row, col]

        if discrete:
            k      = len(np.unique(lbl[lbl >= 0]))
            colors = plt.cm.tab20(np.linspace(0, 1, max(k, 1)))
            noise  = lbl == -1
            if noise.any():
                ax.scatter(umap_xy[noise, 0], umap_xy[noise, 1],
                           c='lightgray', s=4, alpha=0.3,
                           label='Noise', rasterized=True)
            for i in range(k):
                mask = lbl == i
                pct  = 100 * mask.sum() / len(lbl)
                ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
                           color=colors[i], s=4, alpha=0.5,
                           label=f'C{i} ({pct:.0f}%)',
                           rasterized=True)
            if col == 1 and k <= 14:
                ax.legend(fontsize=6, markerscale=2,
                          loc='upper right', ncol=2)
        else:
            # For electrostatic use shared vmin/vmax
            kw = dict(vmin=vmin_e, vmax=vmax_e) \
                 if cbar_label == 'Electrostatic' else {}
            sc = ax.scatter(umap_xy[:, 0], umap_xy[:, 1],
                            c=lbl, cmap=cmap, s=4, alpha=0.5,
                            rasterized=True, **kw)
            if col == 1:
                plt.colorbar(sc, ax=ax, label=cbar_label, shrink=0.7)

        ax.set_title(f'{struct}\n{title}', fontsize=8, fontweight='bold')
        ax.set_xlabel('UMAP 1', fontsize=8)
        ax.set_ylabel('UMAP 2', fontsize=8)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('umap_full_features_corrected.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Comparison summary ────────────────────────────────────────────────────────
print("\n" + "="*65)
print("COMPARISON: ACSF-only vs ACSF + Physicochemical")
print("="*65)
print(f"{'Metric':<40} {'Apo':>10} {'Holo':>10}")
print("-"*65)
print(f"{'HDBSCAN n_clusters (ACSF only)':<40} {'13':>10} {'7':>10}")
print(f"{'HDBSCAN n_clusters (ACSF + physchem)':<40} "
      f"{n_cl_apo_full:>10} {n_cl_holo_full:>10}")
print(f"{'HDBSCAN noise % (ACSF only)':<40} {'38.3%':>10} {'56.8%':>10}")
print(f"{'HDBSCAN noise % (ACSF + physchem)':<40} "
      f"{noise_apo_full:>9.1f}% {noise_holo_full:>9.1f}%")
print(f"{'DBCV (ACSF only)':<40} {'-0.123':>10} {'-0.130':>10}")
print(f"{'DBCV (ACSF + physchem)':<40} "
      f"{dbcv_apo_full:>10.4f} {dbcv_holo_full:>10.4f}")
print("="*65)
print("\nKey question: does adding physicochemical descriptors")
print("reduce noise % or improve DBCV?")
print("If yes → physicochemical adds density structure to the manifold.")
print("If no  → geometry and chemistry encode independent information.")

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ── Use apo for illustration ──────────────────────────────────────────────────
atoms  = calmodulin_apo
maps   = maps1

pos    = atoms.get_positions()
sym    = np.array(atoms.get_chemical_symbols())
cs_xyz = maps.data[['x', 'y', 'z']].values

atom_colors = {'C': '#404040', 'N': '#4169E1',
               'O': '#FF4444', 'S': '#FFD700'}

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Contact space around protein
# ═══════════════════════════════════════════════════════════════════════════════
fig1 = go.Figure()

for elem in sorted(set(sym)):
    mask = sym == elem
    p    = pos[mask]
    fig1.add_trace(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2],
        mode='markers', name=elem,
        marker=dict(size=4, color=atom_colors.get(elem, '#AAAAAA'),
                    opacity=0.85,
                    line=dict(width=0.2, color='black'))))

fig1.add_trace(go.Scatter3d(
    x=cs_xyz[:, 0], y=cs_xyz[:, 1], z=cs_xyz[:, 2],
    mode='markers', name='Contact Space',
    marker=dict(size=2.5, color='dodgerblue', opacity=0.25),
    showlegend=True))

fig1.update_layout(
    title=dict(text='MapSy — Contact Space<br>'
                    '<sup>Calmodulin Apo (1CFD) — '
                    'rotate to find best angle then screenshot</sup>',
               x=0.5, font=dict(size=15)),
    scene=dict(
        aspectmode='data',
        xaxis_title='x (Å)', yaxis_title='y (Å)', zaxis_title='z (Å)',
        xaxis=dict(showticklabels=False, title=''),
        yaxis=dict(showticklabels=False, title=''),
        zaxis=dict(showticklabels=False, title=''),
        bgcolor='white'),
    width=900, height=750,
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, font=dict(size=11)))

fig1.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Selected CS points with cutoff radius spheres
# ═══════════════════════════════════════════════════════════════════════════════
def farthest_point_sample(points, n, seed=7):
    rng      = np.random.default_rng(seed)
    selected = [rng.integers(len(points))]
    for _ in range(n - 1):
        dists = np.array([
            min(np.linalg.norm(points[s] - points[i])
                for s in selected)
            for i in range(len(points))
        ])
        selected.append(int(np.argmax(dists)))
    return np.array(selected)

n_selected   = 5
idx_selected = farthest_point_sample(cs_xyz, n_selected)
selected_pts = cs_xyz[idx_selected]

R_ACSF  = 10.0
R_ELEC  =  8.0
R_HYDRO =  5.0

point_colors = ['#E63946', '#2A9D8F', '#E9C46A', '#9B5DE5', '#F77F00']

def sphere_mesh(center, radius, n=30):
    """Return x, y, z mesh arrays for a sphere surface."""
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(0,     np.pi, n)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones(n), np.cos(v))
    return x, y, z

fig2 = go.Figure()

# Protein atoms
for elem in sorted(set(sym)):
    mask = sym == elem
    p    = pos[mask]
    fig2.add_trace(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2],
        mode='markers', name=elem,
        marker=dict(size=3.5, color=atom_colors.get(elem, '#AAAAAA'),
                    opacity=0.55,
                    line=dict(width=0.2, color='black')),
        showlegend=True))

# All CS points — very transparent background
fig2.add_trace(go.Scatter3d(
    x=cs_xyz[:, 0], y=cs_xyz[:, 1], z=cs_xyz[:, 2],
    mode='markers', name='Contact Space',
    marker=dict(size=2, color='dodgerblue', opacity=0.10),
    showlegend=True))

# For each selected CS point
for i, (pt, col) in enumerate(zip(selected_pts, point_colors)):

    dists = np.linalg.norm(pos - pt, axis=1)

    # Spheres — outermost to innermost so inner ones visible
    for radius, label_r, opacity in [
        (R_ACSF,  f'ACSF {R_ACSF:.0f}Å',   0.04),
        (R_ELEC,  f'Elec {R_ELEC:.0f}Å',   0.07),
        (R_HYDRO, f'Hydro {R_HYDRO:.0f}Å', 0.11),
    ]:
        x, y, z = sphere_mesh(pt, radius, n=30)
        fig2.add_trace(go.Surface(
            x=x, y=y, z=z,
            surfacecolor=np.ones_like(x),
            colorscale=[[0, col], [1, col]],
            opacity=opacity,
            showscale=False,
            name=f'Point {i+1} — {label_r}',
            showlegend=(i == 0),
            hoverinfo='skip'))

    # Atoms inside each radius — colored and sized by which shell
    for radius, size, opa in [
        (R_ACSF,  8, 0.45),
        (R_ELEC,  6, 0.60),
        (R_HYDRO, 4, 0.85),
    ]:
        in_r = dists <= radius
        if in_r.any():
            fig2.add_trace(go.Scatter3d(
                x=pos[in_r, 0], y=pos[in_r, 1], z=pos[in_r, 2],
                mode='markers',
                marker=dict(size=size, color=col, opacity=opa),
                showlegend=False,
                hoverinfo='skip'))

    # Selected CS point — large prominent marker
    fig2.add_trace(go.Scatter3d(
        x=[pt[0]], y=[pt[1]], z=[pt[2]],
        mode='markers',
        name=f'CS point {i+1}',
        marker=dict(size=10, color=col, opacity=1.0,
                    symbol='circle',
                    line=dict(width=2, color='black'))))

fig2.update_layout(
    title=dict(
        text='MapSy — Descriptor Calculation at Contact Space Points<br>'
             '<sup>Nested cutoff radii: '
             f'Hydrophobicity {R_HYDRO:.0f}Å · '
             f'Electrostatic {R_ELEC:.0f}Å · '
             f'ACSF geometry {R_ACSF:.0f}Å — '
             'rotate to best angle then screenshot</sup>',
        x=0.5, font=dict(size=14)),
    scene=dict(
        aspectmode='data',
        xaxis=dict(showticklabels=False, title=''),
        yaxis=dict(showticklabels=False, title=''),
        zaxis=dict(showticklabels=False, title=''),
        bgcolor='white'),
    width=1000, height=800,
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, font=dict(size=10)))

fig2.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

atoms  = calmodulin_apo
maps   = maps1

pos    = atoms.get_positions()
sym    = np.array(atoms.get_chemical_symbols())
cs_xyz = maps.data[['x', 'y', 'z']].values

atom_colors = {'C': '#404040', 'N': '#4169E1',
               'O': '#FF4444', 'S': '#FFD700'}

fig = go.Figure()

for elem in sorted(set(sym)):
    mask = sym == elem
    p    = pos[mask]
    fig.add_trace(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2],
        mode='markers', name=elem,
        marker=dict(size=4, color=atom_colors.get(elem, '#AAAAAA'),
                    opacity=0.85,
                    line=dict(width=0.2, color='black'))))

fig.add_trace(go.Scatter3d(
    x=cs_xyz[:, 0], y=cs_xyz[:, 1], z=cs_xyz[:, 2],
    mode='markers', name='Contact Space',
    marker=dict(size=2.5, color='dodgerblue', opacity=0.25),
    showlegend=True))

fig.update_layout(
    scene=dict(
        aspectmode='data',
        xaxis=dict(showticklabels=False, title=''),
        yaxis=dict(showticklabels=False, title=''),
        zaxis=dict(showticklabels=False, title=''),
        bgcolor='white'),
    width=900, height=750,
    template='plotly_white',
    showlegend=False)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# ── Build feature matrix ──────────────────────────────────────────────────────
rng      = np.random.default_rng(42)
idx_apo  = rng.choice(len(maps1.data), 5000, replace=False)
idx_holo = rng.choice(len(maps2.data), 5000, replace=False)

data_apo_sub  = maps1.data.iloc[idx_apo].reset_index(drop=True)
data_holo_sub = maps2.data.iloc[idx_holo].reset_index(drop=True)

data_apo_sub['electrostatic']  = elec_apo[idx_apo]
data_apo_sub['hydrophobicity'] = hydro_apo[idx_apo]
data_holo_sub['electrostatic'] = elec_holo[idx_holo]
data_holo_sub['hydrophobicity']= hydro_holo[idx_holo]

all_features = list(maps1.features) + ['electrostatic', 'hydrophobicity']

X = np.vstack([
    data_apo_sub[all_features].values,
    data_holo_sub[all_features].values
])
y = np.concatenate([np.zeros(5000), np.ones(5000)])

print(f"Feature matrix: {X.shape}  (54 features, 10000 points, balanced)")

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# ── 5-fold cross validation ───────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_specs = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200,
                                                   n_jobs=4, random_state=42),
    'LDA':                 LinearDiscriminantAnalysis(),
}
colors = {
    'Logistic Regression': 'steelblue',
    'Random Forest':       'darkorange',
    'LDA':                 'green',
}

results_cv  = {name: [] for name in model_specs}
oof_probas  = {name: np.zeros(len(y)) for name in model_specs}

print("\n5-fold cross-validation AUC:")
print("-" * 45)

for fold, (train_idx, test_idx) in enumerate(cv.split(X_sc, y)):
    fold_scaler = StandardScaler().fit(X[train_idx])
    X_train, X_test = fold_scaler.transform(X[train_idx]), fold_scaler.transform(X[test_idx])
    y_train, y_test = y[train_idx],    y[test_idx]

    for name, model in model_specs.items():
        model.fit(X_train, y_train)
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X_test)[:, 1]
        else:
            proba = model.decision_function(X_test)
        oof_probas[name][test_idx] = proba
        results_cv[name].append(roc_auc_score(y_test, proba))

for name, aucs in results_cv.items():
    print(f"  {name:<25} AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

# ── LDA Cohen's d ─────────────────────────────────────────────────────────────
lda_full = LinearDiscriminantAnalysis()
lda_full.fit(X_sc, y)
lda_scores  = lda_full.transform(X_sc).ravel()
scores_apo  = lda_scores[y == 0]
scores_holo = lda_scores[y == 1]
pooled_std  = np.sqrt((scores_apo.std()**2 + scores_holo.std()**2) / 2)
cohend      = abs(scores_apo.mean() - scores_holo.mean()) / pooled_std
print(f"\n  {'LDA Cohen d':<25} d = {cohend:.3f}")

# ── RF feature importance (fit on full data) ──────────────────────────────────
rf_full = RandomForestClassifier(n_estimators=200, n_jobs=4, random_state=42)
rf_full.fit(X_sc, y)
importances = pd.Series(rf_full.feature_importances_, index=all_features)
top10       = importances.nlargest(10).sort_values()

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — ROC curves + feature importance
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── ROC — out-of-fold predictions ────────────────────────────────────────────
ax = axes[0]
for name in model_specs:
    fpr, tpr, _ = roc_curve(y, oof_probas[name])
    auc_mean    = np.mean(results_cv[name])
    auc_std     = np.std(results_cv[name])
    ax.plot(fpr, tpr, lw=2, color=colors[name],
            label=f'{name} (AUC={auc_mean:.3f}±{auc_std:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate',  fontsize=11)
ax.set_title('ROC Curves — Apo vs Holo Classification\n'
             '54 features (52 ACSF + electrostatic + hydrophobicity)\n'
             'Out-of-fold predictions (5-fold CV)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)

# ── Feature importance ────────────────────────────────────────────────────────
ax = axes[1]
ax.barh(range(len(top10)), top10.values, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index, fontsize=9)
ax.set_xlabel('Importance', fontsize=11)
ax.set_title('Top 10 RF Feature Importances\n'
             'Apo vs Holo classification',
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout(pad=3.0)
plt.savefig('roc_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roc_feature_importance.png")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — LDA distribution
# ═══════════════════════════════════════════════════════════════════════════════
fig2, ax2 = plt.subplots(figsize=(8, 5))

ax2.hist(scores_apo,  bins=60, alpha=0.6, color='steelblue',
         label=f'Apo (1CFD)   mean={scores_apo.mean():.2f}',
         density=True)
ax2.hist(scores_holo, bins=60, alpha=0.6, color='firebrick',
         label=f'Holo (1CLL)  mean={scores_holo.mean():.2f}',
         density=True)
ax2.axvline(scores_apo.mean(),  color='steelblue', lw=2, linestyle='--')
ax2.axvline(scores_holo.mean(), color='firebrick',  lw=2, linestyle='--')
ax2.set_xlabel('LDA Score', fontsize=11)
ax2.set_ylabel('Density',   fontsize=11)
ax2.set_title(f"LDA Score Distribution — Cohen's d = {cohend:.3f}\n"
              '54 features (52 ACSF + electrostatic + hydrophobicity)',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: lda_distribution.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LDA COEFFICIENT ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

# Extract LDA coefficients (these define the separating axis)
lda_coefs = pd.Series(lda_full.coef_.ravel(), index=all_features)

# Sort by coefficient value
# Positive coefficients → HOLO-associated (higher values push toward holo)
# Negative coefficients → APO-associated (higher values push toward apo)
lda_coefs_sorted = lda_coefs.sort_values()

print("\n" + "="*60)
print("LDA COEFFICIENT ANALYSIS")
print("="*60)
print("\nPositive coef → feature pushes toward HOLO classification")
print("Negative coef → feature pushes toward APO classification")

print("\n── Top 10 APO-associated features (most negative coefficients) ──")
print(lda_coefs_sorted.head(10).to_string())

print("\n── Top 10 HOLO-associated features (most positive coefficients) ──")
print(lda_coefs_sorted.tail(10).to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE — LDA Coefficients visualization
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: All coefficients ranked
ax = axes[0]
colors_coef = ['firebrick' if c < 0 else 'steelblue' for c in lda_coefs_sorted.values]
ax.barh(range(len(lda_coefs_sorted)), lda_coefs_sorted.values, color=colors_coef, alpha=0.7)
ax.set_yticks(range(len(lda_coefs_sorted)))
ax.set_yticklabels([f.replace('ACSF_', '') for f in lda_coefs_sorted.index], fontsize=6)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('LDA Coefficient', fontsize=11)
ax.set_title('LDA Coefficients — All 54 Features\n'
             'Red = APO-associated | Blue = HOLO-associated',
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Right panel: Top 10 each direction
ax = axes[1]
top_apo = lda_coefs_sorted.head(10)
top_holo = lda_coefs_sorted.tail(10)
combined = pd.concat([top_apo, top_holo])

colors_top = ['firebrick' if c < 0 else 'steelblue' for c in combined.values]
ax.barh(range(len(combined)), combined.values, color=colors_top, alpha=0.8)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels([f.replace('ACSF_', '') for f in combined.index], fontsize=8)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('LDA Coefficient', fontsize=11)
ax.set_title('Top 10 APO-associated & Top 10 HOLO-associated\n'
             'Features driving apo/holo separation',
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('lda_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: lda_coefficients.png")

# ═══════════════════════════════════════════════════════════════════════════════
# Analyze patterns in coefficient signs
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Pattern Analysis ──")

# Parse feature names to extract radius and type
def parse_feature(fname):
    if fname == 'electrostatic':
        return {'type': 'physicochemical', 'radius': 8.0, 'order': None}
    elif fname == 'hydrophobicity':
        return {'type': 'physicochemical', 'radius': 5.0, 'order': None}
    elif '_RS_' in fname:
        # e.g., ACSF_RS_r12.0_n10 → radial structural
        parts = fname.split('_')
        r = float([p for p in parts if p.startswith('r')][0][1:])
        n = int(parts[-1].removeprefix('n'))
        return {'type': 'ACSF_RS', 'radius': r, 'order': n}
    elif '_RC_' in fname:
        # e.g., ACSF_RC_r12.0_n10 → radial compositional
        parts = fname.split('_')
        r = float([p for p in parts if p.startswith('r')][0][1:])
        n = int(parts[-1].removeprefix('n'))
        return {'type': 'ACSF_RC', 'radius': r, 'order': n}
    else:
        return {'type': 'other', 'radius': None, 'order': None}

# Build analysis dataframe
coef_analysis = []
for fname, coef in lda_coefs.items():
    info = parse_feature(fname)
    info['feature'] = fname
    info['coefficient'] = coef
    info['direction'] = 'HOLO' if coef > 0 else 'APO'
    coef_analysis.append(info)

df_coef = pd.DataFrame(coef_analysis)

# Summary statistics
print("\nMean coefficient by feature type and direction:")
print(df_coef.groupby(['type', 'direction'])['coefficient'].agg(['mean', 'count']))

print("\nMean coefficient by radius (ACSF features only):")
acsf_only = df_coef[df_coef['type'].isin(['ACSF_RS', 'ACSF_RC'])]
print(acsf_only.groupby(['radius', 'direction'])['coefficient'].agg(['mean', 'count']))

print("\nMean coefficient by order (ACSF features only):")
print(acsf_only.groupby(['order', 'direction'])['coefficient'].agg(['mean', 'count']))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# ── Predict electrostatic from ACSF geometry alone ───────────────────────────
# Use same subsampled data
scaler_apo_acsf = StandardScaler()
scaler_holo_acsf = StandardScaler()
X_apo_acsf = data_apo_sub[list(maps1.features)].values
X_holo_acsf = data_holo_sub[list(maps1.features)].values

y_elec_apo  = elec_apo[idx_apo]
y_elec_holo = elec_holo[idx_holo]

# Random forest regression: ACSF → electrostatic
rf_reg = RandomForestRegressor(n_estimators=200, n_jobs=4, random_state=42)

r2_apo  = cross_val_score(rf_reg, X_apo_acsf,  y_elec_apo,
                           cv=5, scoring='r2')
r2_holo = cross_val_score(rf_reg, X_holo_acsf, y_elec_holo,
                           cv=5, scoring='r2')

print(f"R² (ACSF → Electrostatic):")
print(f"  Apo:  {r2_apo.mean():.3f}  ± {r2_apo.std():.3f}")
print(f"  Holo: {r2_holo.mean():.3f} ± {r2_holo.std():.3f}")

# Same for hydrophobicity
y_hydro_apo  = hydro_apo[idx_apo]
y_hydro_holo = hydro_holo[idx_holo]

r2h_apo  = cross_val_score(rf_reg, X_apo_acsf,  y_hydro_apo,
                            cv=5, scoring='r2')
r2h_holo = cross_val_score(rf_reg, X_holo_acsf, y_hydro_holo,
                            cv=5, scoring='r2')

print(f"\nR² (ACSF → Hydrophobicity):")
print(f"  Apo:  {r2h_apo.mean():.3f}  ± {r2h_apo.std():.3f}")
print(f"  Holo: {r2h_holo.mean():.3f} ± {r2h_holo.std():.3f}")

In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import numpy as np

# ── Features to plot ──────────────────────────────────────────────────────────
# Top ACSF features + physicochemical
features_to_plot = [
    'electrostatic',
    'hydrophobicity',
    'ACSF_RS_r12.0_001',
    'ACSF_RS_r12.0_004',
    'ACSF_RC_r12.0_003',
    'ACSF_RS_r12.0_005',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, feat in zip(axes, features_to_plot):

    # Get values — physicochemical from arrays, ACSF from dataframes
    if feat == 'electrostatic':
        vals_apo  = elec_apo[idx_apo]
        vals_holo = elec_holo[idx_holo]
    elif feat == 'hydrophobicity':
        vals_apo  = hydro_apo[idx_apo]
        vals_holo = hydro_holo[idx_holo]
    else:
        vals_apo  = data_apo_sub[feat].values
        vals_holo = data_holo_sub[feat].values

    # KS test
    ks_stat, ks_p = ks_2samp(vals_apo, vals_holo)

    # Empirical CDFs
    sorted_apo  = np.sort(vals_apo)
    sorted_holo = np.sort(vals_holo)
    cdf_apo     = np.arange(1, len(sorted_apo)  + 1) / len(sorted_apo)
    cdf_holo    = np.arange(1, len(sorted_holo) + 1) / len(sorted_holo)

    ax.plot(sorted_apo,  cdf_apo,  color='steelblue', lw=2,
            label='Apo (1CFD)')
    ax.plot(sorted_holo, cdf_holo, color='firebrick',  lw=2,
            label='Holo (1CLL)')

    # Find and annotate the KS gap (maximum vertical distance)
    # Interpolate holo CDF at apo points to find max gap
    holo_interp = np.interp(sorted_apo, sorted_holo, cdf_holo)
    gap         = np.abs(cdf_apo - holo_interp)
    max_idx     = np.argmax(gap)
    x_ks        = sorted_apo[max_idx]
    y_apo_ks    = cdf_apo[max_idx]
    y_holo_ks   = np.interp(x_ks, sorted_holo, cdf_holo)

    ax.annotate('', xy=(x_ks, y_holo_ks), xytext=(x_ks, y_apo_ks),
                arrowprops=dict(arrowstyle='<->', color='black',
                                lw=1.5))
    ax.text(x_ks, (y_apo_ks + y_holo_ks) / 2,
            f'  D={ks_stat:.3f}',
            fontsize=9, va='center', color='black')

    # p-value label
    p_str = f'p={ks_p:.2e}' if ks_p > 1e-300 else 'p<1e-300'
    ax.set_title(f'{feat}\nKS={ks_stat:.3f}, {p_str}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Descriptor value', fontsize=9)
    ax.set_ylabel('Cumulative probability', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    'Empirical CDF Comparison — Apo (1CFD) vs Holo (1CLL)\n'
    'KS statistic = maximum vertical distance between CDFs\n'
    'n=5000 subsampled CS points per structure',
    fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('ks_cdf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ks_cdf_comparison.png")

In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Compute KS for all 54 features ───────────────────────────────────────────
ks_results = []

for feat in all_features:
    if feat == 'electrostatic':
        vals_apo  = elec_apo[idx_apo]
        vals_holo = elec_holo[idx_holo]
    elif feat == 'hydrophobicity':
        vals_apo  = hydro_apo[idx_apo]
        vals_holo = hydro_holo[idx_holo]
    else:
        vals_apo  = data_apo_sub[feat].values
        vals_holo = data_holo_sub[feat].values

    ks_stat, ks_p = ks_2samp(vals_apo, vals_holo)
    ks_results.append({
        'feature': feat,
        'ks_stat': ks_stat,
        'ks_p':    ks_p,
        'type':    'Physicochemical' if feat in ['electrostatic', 'hydrophobicity']
                   else ('ACSF_RS' if '_RS_' in feat
                   else ('ACSF_RC' if '_RC_' in feat
                   else 'ACSF_other'))
    })

df_ks = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)

print("Top 15 features by KS statistic:")
print(df_ks.head(15).to_string(index=False))

# ── Color by feature type ─────────────────────────────────────────────────────
type_colors = {
    'Physicochemical': '#E63946',
    'ACSF_RS':         '#2A9D8F',
    'ACSF_RC':         '#E9C46A',
    'ACSF_other':      '#9B5DE5',
}

# ── Figure 1 — Bar plot ranked by KS statistic ───────────────────────────────
fig, ax = plt.subplots(figsize=(18, 8))

colors = [type_colors[t] for t in df_ks['type']]
bars   = ax.bar(range(len(df_ks)), df_ks['ks_stat'],
                color=colors, alpha=0.85, edgecolor='white', linewidth=0.3)

# Significance threshold line (KS critical value at p=0.05 for n=5000)
# Critical value ≈ 1.36 / sqrt(n) for two-sample test
n       = 5000
ks_crit = 1.36 / np.sqrt(n / 2)
ax.axhline(ks_crit, color='black', lw=1.5, linestyle='--', alpha=0.7,
           label=f'D_crit = {ks_crit:.3f} (p = 0.05)')

ax.set_xticks(range(len(df_ks)))
ax.set_xticklabels(
    [f.replace('ACSF_', '') for f in df_ks['feature']],
    rotation=90, fontsize=7)
ax.set_ylabel('KS Statistic (D)', fontsize=11)
ax.set_title('KS Statistics — All 54 Features\n'
             'Apo (1CFD) vs Holo (1CLL), n=5000 CS points per structure',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Legend for feature types
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.85, label=t)
                   for t, c in type_colors.items()
                   if t in df_ks['type'].values]
legend_elements.append(
    plt.Line2D([0], [0], color='black', lw=1.5, linestyle='--',
               label=f'p=0.05 threshold'))
ax.legend(handles=legend_elements, fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('ks_all_features_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2 — CDF comparison for top 6 + bottom 2 ───────────────────────────
# Top 6 most discriminative + 2 least to show contrast
top6   = df_ks.head(6)['feature'].tolist()
bottom2= df_ks.tail(2)['feature'].tolist()
to_plot= top6 + bottom2

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.ravel()

for ax, feat in zip(axes, to_plot):
    if feat == 'electrostatic':
        vals_apo  = elec_apo[idx_apo]
        vals_holo = elec_holo[idx_holo]
    elif feat == 'hydrophobicity':
        vals_apo  = hydro_apo[idx_apo]
        vals_holo = hydro_holo[idx_holo]
    else:
        vals_apo  = data_apo_sub[feat].values
        vals_holo = data_holo_sub[feat].values

    ks_stat, ks_p = ks_2samp(vals_apo, vals_holo)
    row = df_ks[df_ks['feature'] == feat].iloc[0]

    sorted_apo  = np.sort(vals_apo)
    sorted_holo = np.sort(vals_holo)
    cdf_apo     = np.arange(1, len(sorted_apo)  + 1) / len(sorted_apo)
    cdf_holo    = np.arange(1, len(sorted_holo) + 1) / len(sorted_holo)

    col = type_colors[row['type']]
    ax.plot(sorted_apo,  cdf_apo,  color='steelblue', lw=2,
            label='Apo (1CFD)')
    ax.plot(sorted_holo, cdf_holo, color='firebrick',  lw=2,
            label='Holo (1CLL)')

    # KS gap annotation
    holo_interp = np.interp(sorted_apo, sorted_holo, cdf_holo)
    gap         = np.abs(cdf_apo - holo_interp)
    max_idx     = np.argmax(gap)
    x_ks        = sorted_apo[max_idx]
    y_apo_ks    = cdf_apo[max_idx]
    y_holo_ks   = np.interp(x_ks, sorted_holo, cdf_holo)
    ax.annotate('', xy=(x_ks, y_holo_ks), xytext=(x_ks, y_apo_ks),
                arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
    ax.text(x_ks, (y_apo_ks + y_holo_ks) / 2,
            f'  D={ks_stat:.3f}', fontsize=9, va='center')

    p_str = f'p={ks_p:.2e}' if ks_p > 1e-300 else 'p<1e-300'
    rank  = df_ks[df_ks['feature'] == feat].index[0] + 1
    ax.set_title(f'{feat.replace("ACSF_", "")}\n'
                 f'KS={ks_stat:.3f}, {p_str}',
                 fontsize=9, fontweight='bold',
                 color=col)
    ax.set_xlabel('Descriptor value', fontsize=8)
    ax.set_ylabel('Cumulative probability', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    'CDF Comparison — Top 6 Most Discriminative + 2 Least Discriminative Features\n'
    'Apo (1CFD) vs Holo (1CLL), n=5000 CS points per structure',
    fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('ks_cdf_top_bottom.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: ks_all_features_bar.png, ks_cdf_top_bottom.png")

In [ ]:
import json
checks = {'aperiodic_apo': not bool(calmodulin_apo.pbc.any()), 'aperiodic_holo': not bool(calmodulin_holo.pbc.any()), 'ac_channels_apo': len(maps1.features), 'ac_channels_holo': len(maps2.features), 'points_apo': len(maps1.data), 'points_holo': len(maps2.data)}
assert checks['ac_channels_apo'] == checks['ac_channels_holo'] == 52
for tag, m in [('apo', maps1), ('holo', maps2)]:
    values = m.data[list(m.features) + ['electrostatic','hydrophobicity']].to_numpy(dtype=float)
    assert values.shape[1] == 54 and np.isfinite(values).all()
    checks[tag + '_finite'] = True
assert checks['aperiodic_apo'] and checks['aperiodic_holo']
(OUTPUT / 'scientific_checks.json').write_text(json.dumps(checks, indent=2))
metric_rows = [{'state':tag, 'k':r['k'], 'silhouette':r['sil'], 'davies_bouldin':r['db']} for tag in ['apo','holo'] for r in results[tag]]
pd.DataFrame(metric_rows).to_csv(OUTPUT / 'clustering_metrics.csv', index=False)
print(checks)
